In [ ]:
#Colab optioneel libraries installeren
#%pip install -q pandas numpy pyarrow matplotlib seaborn plotly dash squarify requests paramiko

# Importeren van libraries

In [ ]:
#Importeren van libraries
import pandas as pd
#Pandas om dataframes te maken en analyseren
import pyarrow
#Pyarrow module om .csv's na cleaning en validatie in parquet-formaat op te slaan.
from pathlib import Path
#Path om binnen de omgeving .csv-files te zoeken en lade n.
import re
#Regex module importeren om pattern matching te gebruiken.
import numpy as np
#Numpy module voor matrix berekeningen. Voornamelijk gebruikt om vergelijkingen te maken met np.isclose.


#Modules om downloaden en uploaden van bestanden naar eigen domein mogelijk te maken.
import requests
# Versturen van HTTP(S)-verzoeken om CSV- en Parquet-bestanden te downloaden.
import paramiko
# Opzetten van beveiligde SSH/SFTP-verbindingen voor het uploaden van bestanden.
from getpass import getpass
# Veilig invoeren van wachtwoorden zonder deze zichtbaar te tonen of in de code op te slaan.
from uuid import uuid4
# Genereren van unieke bestandsnamen voor tijdelijke bestanden tijdens uploads.


pd.options.display.float_format = "{:,.2f}".format
#Floats weergeven als getallen met 2 decimalen achter de komma, aanvankelijk neigt pandas bij grotere getallen naar wetenschappelijke weergaven.

# Functies

In [156]:
#Functie om .csv en parquet bestanden te downloaden en verwerken vanaf eigen domein. Geschreven met behulp van ChatGPT, gezien dit niet expliciet onderdeel was van de opdracht.
#Er voor gekozen dit toe te voegen om te garanderen dat de gehele pipeline met dezelfde data werkt en te voorkomen dat scripts niet werken vanwege lokaal draaien.
#Voor de online versie zijn de .csv bestanden van Contoso op een eigen domein gehost, zodat deze ingeladen kunnen worden via bijvoorbeeld google colab.
#Deze functie haalt de gegevens op van dit domein en laad ze in het notebook voor verdere analyse en verwerking.

# Serverinstellingen
BASE_URL = "https://kilometerapp.nl/contoso-data"
USERNAME = "contoso_reader"
PASSWORD = getpass("Voer het SFTP-wachtwoord in: ")

BASE_URL = "https://kilometerapp.nl/contoso-data"

def load_data(filename, **kwargs):
    extension = Path(filename).suffix.lower()

    if extension == ".csv":
        url = f"{BASE_URL}/{filename}"
        folder = "data"
    elif extension == ".parquet":
        url = f"{BASE_URL}/PQ/{filename}"
        folder = "PQ"
    else:
        raise ValueError("Alleen CSV en Parquet worden ondersteund")

    local_dir = Path("/content/contoso") / folder
    local_dir.mkdir(parents=True, exist_ok=True)
    local_path = local_dir / filename

    with requests.get(
        url,
        auth=(USERNAME, PASSWORD),
        stream=True,
        timeout=120
    ) as response:
        response.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    if extension == ".csv":
        return pd.read_csv(local_path, **kwargs)

    return pd.read_parquet(local_path, **kwargs)


In [39]:
#Functie om parquet bestanden na bewerking te uploaden naar eigen domein. Geschreven met behulp van ChatGPT, gezien dit niet expliciet onderdeel was van de opdracht.
#Er voor gekozen dit toe te voegen om te garanderen dat de gehele pipeline met dezelfde data werkt en te voorkomen dat scripts niet werken vanwege lokaal draaien.
HOST = "37.97.169.50"
USER = "contoso_uploader"

LOCAL_DIR = Path("PQ")
REMOTE_DIR = "/PQ"


def upload_parquet(df, filename, sftp):
    """Exporteer een DataFrame en vervang het Parquet-bestand op de VPS."""

    if Path(filename).name != filename or not filename.endswith(".parquet"):
        raise ValueError("Ongeldige Parquet-bestandsnaam")

    # 1. Lokaal exporteren
    LOCAL_DIR.mkdir(exist_ok=True)
    local_path = LOCAL_DIR / filename

    df.to_parquet(local_path, index=False)

    # 2. Serverpaden
    remote_path = f"{REMOTE_DIR}/{filename}"
    temp_path = f"{REMOTE_DIR}/.{filename}.{uuid4().hex}.tmp"

    try:
        # 3. Eerst tijdelijk uploaden
        sftp.put(str(local_path), temp_path)

        # Controleren of de upload volledig is
        if sftp.stat(temp_path).st_size != local_path.stat().st_size:
            raise IOError("Bestandsgrootte komt niet overeen")

        # Leesrechten voor Nginx
        sftp.chmod(temp_path, 0o644)

        # 4. Bestaand bestand atomair vervangen
        sftp.posix_rename(temp_path, remote_path)

    except Exception:
        try:
            sftp.remove(temp_path)
        except OSError:
            pass
        raise

    print(f"Succesvol geüpload: {filename}")
    print(f"Rijen: {len(df):,}")
    print(f"Bestandsgrootte: {local_path.stat().st_size / 1024**2:.2f} MB")

In [40]:
#Functie voor initi"le analyse
def init_analyse(df) -> None:
    print("informatie over dataframe:")
    df.info()
    #.info geeft informatie over het aantal records, kolommen en de datatypes.
    # Geeft een goed eerste idee van de dataset en eventuele afwijkingen in de verwachtingen.
    print()
    print(f"aantal gedupliceerde rijen: {df.duplicated().sum()}")
    #df. duplicated controleert de dataset op rijen binnen de set die exact dezelde waarden hebben.
    print()
    print("Overzicht van lege waarden:")
    display(df.isna().sum())
    #df.isna controleert alle kolommen binnen de dataframe op lege waarden. 
    print()
    print("Aantal unieke waarden:")
    display(df.nunique())
    print()
    print("Eerste 20 rijen:")
    display(df.head(20))
    #Laat de eerste 20 records zien, geeft een beeld van hoe data wordt weergeven en samenhangt.
    print()
    print("Laatste 20 rijen:")
    display(df.tail(20))
    #Hetzelfde als head, maar dan voor de laatste 20 records. Afhankelijk van sortering kan dit andere inzichten geven.

# 1 ***Analyse sales data***

In dit onderdeel importeren en analyseren we de salesdata met behulp van pandas

## 1.1 Importeren en inlezen dataset

In [157]:
#Importeren van contoso sales data.
sales_df= load_data("Contoso_Sales.csv",
                      decimal=",")

Data is met behulp van pathlib en pandas ingelezen.  
Bij het inlezen ontstond initieel een dtypewarning voor de kolommen ReturnAmount en DiscountAmount. 
Door bij het inlezen het decimal teken mee te geven wordt deze fout verholpen en worden floats gelijk correct ingelezen.

## 1.2. Analyseren sales dataset
Hier wordt de algehele dataset geanalyseerd. Eerst wordt gekeken naar aantal kolommen en de data die de regels bevatten. Vervolgens naar eventuele afwijkingen. 

In [42]:
sales_df.info()
#Eerste data over kolommen opvragen

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   SalesKey          int64  
 1   DateKey           str    
 2   channelKey        int64  
 3   StoreKey          int64  
 4   ProductKey        int64  
 5   PromotionKey      int64  
 6   UnitCost          float64
 7   UnitPrice         float64
 8   SalesQuantity     int64  
 9   ReturnQuantity    int64  
 10  ReturnAmount      float64
 11  DiscountQuantity  int64  
 12  DiscountAmount    float64
 13  TotalCost         float64
 14  SalesAmount       float64
dtypes: float64(6), int64(8), str(1)
memory usage: 311.3 MB


- Datekey wordt als string weergegeven, deze kan nog omgezet te worden naar een datum datatype.
- UnitCost, UnitPrice, TotalCost en SalesAmount werden initieel als strings geïnterpreteerd. Dit is afgevangen door bij de import de parameter decimal mee te geven en komma's aan te merken als het decimaal-teken.

In [43]:
sales_df.head(20)
#De eerste 20 rijn tonen met waarden.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,"2011-05-07 00:00:00,000",1,76,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
1,26527,"2013-05-22 00:00:00,000",1,26,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
2,26702,"2013-09-30 00:00:00,000",1,298,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
3,33125,"2012-10-31 00:00:00,000",1,264,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
4,52306,"2013-06-26 00:00:00,000",1,98,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
5,67230,"2012-10-28 00:00:00,000",1,194,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
6,79246,"2012-05-12 00:00:00,000",1,230,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
7,87210,"2013-04-17 00:00:00,000",1,84,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
8,91129,"2013-05-26 00:00:00,000",1,215,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
9,109148,"2013-04-06 00:00:00,000",1,93,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"


Datekey lijkt een datumwaarde te moeten zijn. Op het moment is dit een string. Deze dient bij het opschonen omgezet te worden, zodat deze als key gebruikt kan worden.  
UnitCost, UnitPrice, TotalCost en SalesAmount lijkt om bedragen te gaan. De UnitPricemaal de SalesQuantity is hier  gelijk aan de Totalcost en SalesAmount.   
ChannelKey is niet in constante camelcase geschreven met de rest van de set.

In [44]:
sales_df.tail(20)
#Laatste 20 waarden bekijken.

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
2282462,3366917,"2011-06-28 00:00:00,000",1,120,613,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282463,3367028,"2011-05-20 00:00:00,000",1,12,320,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282464,3369192,"2012-05-15 00:00:00,000",1,60,320,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282465,3369889,"2013-04-29 00:00:00,000",1,167,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282466,3369908,"2012-10-06 00:00:00,000",1,115,405,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282467,3370934,"2012-04-21 00:00:00,000",1,142,624,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282468,3371111,"2013-05-23 00:00:00,000",1,21,345,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282469,3372220,"2011-05-29 00:00:00,000",1,2,613,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282470,3378122,"2011-06-25 00:00:00,000",1,243,405,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282471,3378463,"2013-10-26 00:00:00,000",1,282,345,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"


De laatste 20 rijen laten hetzelfde patroon zien. In deze rijen zitten ook returns verwerkt. Deze lijken wel goed verwerkt te worden in de TotalCost, bijvoorbeeld 10 items verkocht - 1 return = 9 * UnitCost.  
Op salesniveau lijkt dit echter niet correct doorberekend. De SalesAmount blijft gelijk aan 10 maal de UnitPrice, ondanks de return.  
Voor goede vergelijkende data op dit vlak is het belangrijk dat zowel de omzet(SalesAmount) als kosten(TotalCost) op dezelfde manier worden berekend.  
De beste actie lijkt  om een extra kolom toe te voegen die de SalesAmount berekend met inachtname van Returns.    
Dit houdt de originele kolom in tact, maar geeft een betere representatie van de daadwerkelijke omzet van een order.

In [45]:
sales_df.describe()

,SalesKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
count,"2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00","2,282,482.00"
mean,"1,703,424.16",1.63,196.38,881.19,8.50,124.60,297.18,16.17,0.14,42.56,1.48,55.52,"1,556.38","3,654.45"
std,"983,152.96",1.03,95.54,509.48,8.26,126.75,347.31,39.35,0.36,172.02,1.80,139.05,"1,888.55","4,853.79"
min,1.00,1.00,1.00,1.00,1.00,0.48,0.95,4.00,0.00,0.00,0.00,0.00,1.92,3.04
25%,"851,809.50",1.00,121.00,453.00,1.00,30.58,66.00,9.00,0.00,0.00,0.00,0.00,321.48,658.42
50%,"1,704,423.50",1.00,200.00,880.00,4.00,87.37,200.00,10.00,0.00,0.00,1.00,2.42,986.04,"2,293.60"
75%,"2,554,564.75",2.00,296.00,"1,273.00",14.00,162.64,358.00,13.00,0.00,0.00,3.00,53.98,"2,065.23","4,699.70"
max,"3,406,089.00",4.00,310.00,"2,517.00",28.00,960.82,"2,899.99","2,880.00",5.00,"9,996.00",24.00,"6,497.40","84,115.20","235,467.00"


In [46]:
sales_df.isna().sum()
#Controleren op lege waarden in cellen.


SalesKey            0
DateKey             0
channelKey          0
StoreKey            0
ProductKey          0
PromotionKey        0
UnitCost            0
UnitPrice           0
SalesQuantity       0
ReturnQuantity      0
ReturnAmount        0
DiscountQuantity    0
DiscountAmount      0
TotalCost           0
SalesAmount         0
dtype: int64

In [47]:
sales_df.nunique() 
#Aantal unique waarden opsommen.

SalesKey            2282482
DateKey                1096
channelKey                4
StoreKey                306
ProductKey             1689
PromotionKey             28
UnitCost                397
UnitPrice               350
SalesQuantity            98
ReturnQuantity            6
ReturnAmount            782
DiscountQuantity         23
DiscountAmount         8572
TotalCost             14114
SalesAmount           51717
dtype: int64

In [48]:
sales_df[sales_df.duplicated()]
#Checken op duplicated rows

,SalesKey,DateKey,channelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount


Op basis van bovenstaande analyses zijn er geen lege waardes of duplicaten aanwezig in deze dataset.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 verkoopkanalen volgens sales set.
- 306 unieke winkels volgens sales set.
- 1689 soort producten
- 28 promoties

### **Tussentijdse bevindingen**

De sales dataset bestaat uit ***2.282.482*** unieke saleskeys.  
Er zal een aparte waarden gemaakt dienen te worden die return amounts verrekend met de SalesAmount, gezien het huidige verschil tussen kosten en opbrengsten bij een retour.  

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 **verkoopkanalen** volgens sales set.
- 306 **unieke winkels** volgens sales set.
- 1689 soort **producten**
- 28 **promoties**

Een eerste run van de validatie leverde een error op bij het controleren van het verband tussen UnitCost, SalesQuantity en ReturnQuantity.   
Uit de tail data uit de initiële analyse leek het logisch om aan te nemen dat alle TotalCost waarden waren berekend volgens de logica:  *SalesQuantity * UnitCost - ReturnQuantity * UnitCost = TotalCost*.
Voor deze 20 rijen ging dit ook op. De validatie toonde echter aan dat dit voor een groot deel van de set niet geld.


*To-do*:  
- DateKey omzetten naar datetime dtype om mee te rekenen.
- Aanpassen camelcase; "channelkey" -> "ChannelKey".
- Controleren verband tussen SalesAmount en ReturnAmount.
- Controleren verband tussen TotalCost en ReturnAmount.

## 1.3 Data cleaning

Op basis van de analyse zien we dat de datum gegevens nu niet als juiste dtype worden ingegeven.  
In dit onderdeel worden deze omgezet naar bruikbaar datums.

### Datekey omzetten naar datetime waarde.

In [49]:
#Dataframe kopieeëren als backup. Zo blijft de originele dataset altijd behouden ter controle. 
sales_df2 = sales_df.copy()

In [50]:
#channelKey veranderen naar ChannelKey voor consistente camelcase.
sales_df2 =sales_df2.rename(columns={"channelKey":"ChannelKey"})

In [51]:
#Omzetten naar datetime van DateKey. De originele string bevat gegevens tot en met microseconden dus dient hier rekening mee gehouden worden bij de conversie.
sales_df2["DateKey"] = pd.to_datetime(sales_df2["DateKey"], format="%Y-%m-%d %H:%M:%S,%f")
display(sales_df2)

,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount
0,2201,2011-05-07,1,76,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
1,26527,2013-05-22,1,26,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
2,26702,2013-09-30,1,298,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
3,33125,2012-10-31,1,264,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
4,52306,2013-06-26,1,98,375,1,321.44,699.00,10,0,0.00,0,0.00,"3,214.40","6,990.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,2013-05-15,1,40,397,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282478,3395082,2013-10-04,1,52,565,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282479,3396997,2011-07-26,1,303,381,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"
2282480,3398163,2013-07-04,1,247,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00"


In [52]:
#Kijken naar begin en einddatum van dataset.
print(sales_df2.loc[sales_df2["DateKey"].idxmin(),"DateKey"])
print(sales_df2.loc[sales_df2["DateKey"].idxmax(),"DateKey"])

2011-01-01 00:00:00
2013-12-31 00:00:00


De dataset loopt van 1 januari 2011 tot en met 31 december 2013.

In [53]:
#Nieuwe sales dataframe met date als index, zodat resamplen mogelijk is.
datesales = sales_df2.set_index("DateKey")
datesales.sort_values(by="DateKey")

#Sorteren van nieuwe naar oud.
SalesByMonth = datesales.resample("ME").agg(TotalKeys=("SalesKey","nunique"),
                             VerkoopKanalen=("ChannelKey","nunique"),
                             TotalCost=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum"))


#Dataframe die kijkt naar aantal verkopen, kanalen, totale kosten en totale omzet per maand.
SalesByMonth["Brutomarge"] = SalesByMonth["SalesAmount"] - SalesByMonth["TotalCost"]
#Kolom Brutomarge toevoegen aan dataframe.

SalesByMonth["BrutomargePct"] = SalesByMonth["Brutomarge"]/SalesByMonth["SalesAmount"]
#Brutomarge uitdrukken als percentage van de totale omzet.

datesales.sort_values(by="DateKey")
#Sorteren van nieuwe naar oud.
SalesByMonth = datesales.resample("ME").agg(TotalKeys=("SalesKey","nunique"),
                             VerkoopKanalen=("ChannelKey","nunique"),
                             TotalCost=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum"))


#Dataframe die kijkt naar aantal verkopen, kanalen, totale kosten en totale omzet per maand.
SalesByMonth["Brutomarge"] = SalesByMonth["SalesAmount"] - SalesByMonth["TotalCost"]
#Kolom Brutomarge toevoegen aan dataframe.

SalesByMonth["BrutomargePct"] = SalesByMonth["Brutomarge"]/SalesByMonth["SalesAmount"]
#Brutomarge uitdrukken als percentage van de totale omzet.

display(SalesByMonth)
#Tussentijds inzicht in de data, niet per se nodig hier.

,TotalKeys,VerkoopKanalen,TotalCost,SalesAmount,Brutomarge,BrutomargePct
DateKey,,,,,,
2011-01-31,73868,4,"81,593,589.36","193,305,554.64","111,711,965.28",0.58
2011-02-28,70212,4,"88,412,165.20","209,439,067.93","121,026,902.73",0.58
2011-03-31,73596,4,"86,905,506.38","203,991,979.69","117,086,473.31",0.57
2011-04-30,91338,4,"116,872,052.05","276,891,048.16","160,018,996.11",0.58
2011-05-31,94223,4,"122,236,876.68","288,749,508.61","166,512,631.93",0.58
2011-06-30,92553,4,"119,806,180.24","283,186,644.54","163,380,464.30",0.58
2011-07-31,76193,4,"115,362,984.04","272,818,635.11","157,455,651.07",0.58
2011-08-31,76085,4,"111,509,275.12","263,780,279.28","152,271,004.16",0.58
2011-09-30,74802,4,"108,858,863.84","257,282,781.95","148,423,918.11",0.58


De Datekey is succesvol omgezet in een bruikbaar datumtype voor toekomstige analyses.  
Eerste analyse laat vooral een teruglopend aantal verkopen zien over de periode van 1 januari 2011 tot en met 31 december 2013.  
Omzetten en kosten lopen samenhangend hiermee terug.

### Verband tussen SalesAmount en ReturnAmount

In [54]:
#Aparte kolom maken waarin returns worden verrekend met de omzet. Naast de returnquantity dient ook de discount verrekend te worden, deze werd eerder al wel correct verrekend.
sales_df2["SalesAmountRet"] = (sales_df2["SalesQuantity"]*sales_df2["UnitPrice"])-(sales_df2["ReturnQuantity"]*sales_df2["UnitPrice"])-sales_df["DiscountAmount"]

#Controleren of aanname aansluit, meer dan de helft van de waarden komt terug als false.
display((sales_df2["SalesAmount"]-sales_df2["ReturnAmount"] == sales_df2["SalesAmountRet"]).value_counts())


True     2031285
False     251197
Name: count, dtype: int64

Voor 251.197 van de regels lijkt de logica achter de berekening van de SalesAmountRet waarden niet te kloppen.

In [55]:
#Regels bekijken waar SalesAmountRet niet gelijk is aan SalesAmount minus ReturnAmount.
display(sales_df2[~(sales_df2["SalesAmountRet"] == sales_df2["SalesAmount"]-sales_df2["ReturnAmount"])])

,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet
24755,32341,2011-06-23,1,52,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90
24756,44300,2011-05-05,1,66,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90
24757,62076,2011-06-06,1,26,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90
24758,73426,2011-05-10,1,184,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90
24759,88587,2011-06-09,1,119,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282346,3405335,2011-05-06,1,3,1359,1,14.24,42.99,10,1,42.99,0,0.00,128.16,429.90,386.91
2282348,3405398,2011-06-08,1,123,1395,1,7.81,16.99,10,1,16.99,0,0.00,70.29,169.90,152.91
2282351,3405618,2011-10-22,1,253,1385,1,10.57,22.99,10,1,22.99,0,0.00,95.13,229.90,206.91
2282356,3405833,2011-05-12,1,22,1358,1,18.39,39.99,10,1,39.99,0,0.00,165.51,399.90,359.91


De gepresenteerde regels lijken ook gelijk te zijn aan het eerdere verband, vermoedelijk zit het verschil in de afronding.

In [56]:
#Copy maken van sales_df2 om te controleren of SalesAmount met returns en discounts gelijk is aan SalesAmounts met returns.
salescontrole = sales_df2.copy()
#Kolom toevoegen die controleert of SalesAmount met returns gelijk is aan SalesAmount + Returns.
salescontrole["Check"] = salescontrole["SalesAmountRet"] - salescontrole["SalesAmount"]+ salescontrole["ReturnAmount"]
#Controleren op waarden die niet gelijk zijn aan 0. Deze controle geeft nog veel rijen, ook al is de controle 0, vermoeden is dat dit afrondingsverschillen zijn.
display(salescontrole[salescontrole["Check"]!=0])
#Controleren of er waarden zijn waarvoor het voorgaande niet klopt, waarbij afrondingsverschillen van 1 decimaal worden uitgesloten door .abs()>0.000001.
#Deze bevat geen waarden dus de aanname klopt. 
display(salescontrole[salescontrole["Check"].abs()>0.000001])


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,Check
24755,32341,2011-06-23,1,52,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90,0.00
24756,44300,2011-05-05,1,66,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90,0.00
24757,62076,2011-06-06,1,26,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90,0.00
24758,73426,2011-05-10,1,184,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90,0.00
24759,88587,2011-06-09,1,119,1356,1,16.55,35.99,10,0,0.00,0,0.00,165.50,359.90,359.90,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282351,3405618,2011-10-22,1,253,1385,1,10.57,22.99,10,1,22.99,0,0.00,95.13,229.90,206.91,-0.00
2282353,3405652,2011-05-18,1,59,1641,1,5.82,12.66,10,1,12.66,0,0.00,52.38,126.60,113.94,0.00
2282356,3405833,2011-05-12,1,22,1358,1,18.39,39.99,10,1,39.99,0,0.00,165.51,399.90,359.91,0.00
2282357,3405840,2012-05-26,1,45,773,1,8.11,15.90,10,1,15.90,0,0.00,72.99,159.00,143.10,-0.00


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,Check


De laatste controle bevestigt dat de False waarden werden veroorzaakt door een afwijking in de afronding. 
Bij een afronding van van 0.000001 zijn er geen afwijkingen meer.

### Verband tussen TotalCost en ReturnAmount

In [57]:
#Conform de SalesAmountRet logica een nieuwe kolom creeëren die de totale kosten van een regel berekend met inachtname van de returns.
sales_df2["TotalCostRet"] = (sales_df2["SalesQuantity"]*sales_df2["UnitCost"]) - (sales_df2["ReturnQuantity"]*sales_df["UnitCost"])

#Masker om te controleren welke waarde gelijk zijn aan de berekening met een afwijking van 0.001.
costcontrole = np.isclose(sales_df2["TotalCostRet"],
                          sales_df2["TotalCost"],
                            atol=0.001)

#Eerder gemaakt masker omdraaien om rijen te weergeven met waarden waar de vergelijking niet klopt. Sort_values by DateKey om te inspecteren of dit het geval is voor een bepaalde datumreeks.
display(sales_df2[~costcontrole].sort_values(by="DateKey"))


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,TotalCostRet
973369,940885,2011-01-01,1,86,1269,2,25.47,49.96,4,1,49.96,4,9.99,101.88,189.85,139.89,76.41
2014672,1815130,2011-01-01,1,246,550,10,55.57,109.00,4,1,109.00,3,65.40,222.28,370.60,261.60,166.71
953193,19380,2011-01-01,1,150,1510,2,65.77,129.00,4,1,129.00,1,6.45,263.08,509.55,380.55,197.31
2271514,3354016,2011-01-01,1,251,1468,5,86.91,189.00,6,1,189.00,0,0.00,521.46,"1,134.00",945.00,434.55
2006972,2320463,2011-01-01,1,282,801,2,10.15,19.90,4,1,19.90,2,1.99,40.60,77.61,57.71,30.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950918,587210,2013-12-31,1,273,177,23,29.01,56.90,26,1,56.90,1,8.54,754.26,"1,470.87","1,413.96",725.25
981786,47027,2013-12-31,1,292,526,23,65.77,129.00,26,1,129.00,0,0.00,"1,710.02","3,354.00","3,225.00","1,644.25"
2009331,1639400,2013-12-31,2,307,525,23,50.47,99.00,26,1,99.00,2,29.70,"1,312.22","2,544.30","2,445.30","1,261.75"
1960032,3106856,2013-12-31,1,275,1182,23,516.86,"1,560.00",12,1,"1,560.00",3,702.00,"6,202.32","18,018.00","16,458.00","5,685.46"


Door de gehele dataset zitten waarden, waarbij de eerdere aanname van de TotalCost berekening niet aansluit, namelijk dat de returns hier al in verwerkt zaten.
Volgende aanname is: *TotalCost - (SalesQuantity * UnitCost) = TotalCostRet*.

In [58]:
masker = np.isclose(sales_df2["TotalCostRet"],
                    sales_df2["TotalCost"]-(sales_df2["UnitCost"]*sales_df2["ReturnQuantity"]),
                      atol=0.001,
                      rtol=0)
#masker om te controleren waar de aanname klopt.
display(sales_df2[~masker])
display(sales_df2.loc[masker &(sales_df2["ReturnQuantity"]>0)])

,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,TotalCostRet
902431,3408,2012-09-20,1,38,1498,12,137.96,300.00,12,1,300.00,1,30.00,"1,517.56","3,570.00","3,270.00","1,517.56"
902432,4722,2012-07-06,1,146,1319,12,10.57,22.99,12,1,22.99,1,2.30,116.27,273.58,250.59,116.27
902433,5333,2012-07-11,1,48,1357,12,17.93,38.99,12,1,38.99,1,3.90,197.23,463.98,424.99,197.23
902434,5947,2012-07-15,1,9,402,12,430.38,"1,299.00",12,1,"1,299.00",1,129.90,"4,734.18","15,458.10","14,159.10","4,734.18"
902435,7878,2012-09-27,1,50,468,12,82.32,179.00,12,1,179.00,1,17.90,905.52,"2,130.10","1,951.10",905.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2282477,3392414,2013-05-15,1,40,397,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00","6,291.00","2,892.96"
2282478,3395082,2013-10-04,1,52,565,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00","6,291.00","2,892.96"
2282479,3396997,2011-07-26,1,303,381,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00","6,291.00","2,892.96"
2282480,3398163,2013-07-04,1,247,201,1,321.44,699.00,10,1,699.00,0,0.00,"2,892.96","6,990.00","6,291.00","2,892.96"


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,TotalCostRet
906476,89570,2011-08-27,2,199,483,3,50.47,99.00,12,1,99.00,4,39.60,605.64,"1,148.40","1,049.40",555.17
906478,233854,2011-07-14,2,199,538,3,50.47,99.00,12,1,99.00,4,39.60,605.64,"1,148.40","1,049.40",555.17
906479,264150,2011-08-23,2,199,121,3,61.17,119.99,12,1,119.99,4,48.00,734.04,"1,391.88","1,271.89",672.87
906482,292247,2011-09-06,2,199,213,3,152.90,299.90,12,1,299.90,4,119.96,"1,834.80","3,478.84","3,178.94","1,681.90"
906483,302283,2011-09-16,2,199,9,3,30.58,59.99,12,1,59.99,4,24.00,366.96,695.88,635.89,336.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2281482,3338948,2011-05-20,1,15,904,1,38.74,75.99,10,1,75.99,0,0.00,387.40,759.90,683.91,348.66
2281868,3369327,2011-07-25,1,213,1238,1,85.65,168.00,10,1,168.00,0,0.00,856.50,"1,680.00","1,512.00",770.85
2282092,3386069,2011-10-31,1,269,67,1,13.10,25.69,10,1,25.69,0,0.00,131.00,256.90,231.21,117.90
2282314,3402648,2011-04-21,1,87,844,1,13.71,26.90,10,1,26.90,0,0.00,137.10,269.00,242.10,123.39


Uit de controle blijkt dat de berekeningswijze van TotalCost inconsistent is. Bij sommige transacties wordt ReturnQuantity meegenomen in de berekening van de kostprijs, terwijl dit bij andere transacties niet gebeurt.  
Deze inconsistentie kan leiden tot onjuiste uitkomsten bij verdere analyses, bijvoorbeeld bij het berekenen van de brutomarge van producten.  
Om zuivere vergelijkingen tussen omzet en kosten te kunnen maken wordt hiervoor in het vervolg de TotalCostRet kolom gehanteerd. 

### Samenvatting
De DateKey is omgezet in een datumwaarde.  
ChannelKey is hernoemd van "channelKey" naar  "ChannelKey", zodat deze consistent is met de rest van de set.  
Er is een kolom toegevoegd, SalesAmountRet, die omzet berekend met inachtname van retourproducten en kortingen.  
Er is een kosten kolom, TotalCostRet, toegevoegd die consequent rekening houdt met returns constante wijze is berekend.

## 1.4 Validatie
Hier worden eerdere aannames en verbanden gevalideerd. Een error op één van de onderdelen leidt tot het stoppen van het notebook, waarbij niet geëscaleerd wordt naar de volgende fase.  
Er wordt een AssertionEror met bijbehorend bericht gegenereerd waar de validatie faalt.

In [59]:
assert sales_df2["SalesKey"].is_unique, "SalesKey bevat dubbele waarden"
# Controleert of alle SalesKeys uniek zijn.
# Bij dubbele waarden faalt de validatie met een AssertionError.

assert sales_df2["SalesKey"].notna().all(), "SalesKey bevat ontbrekende waarde"
# Controleert of iedere rij een SalesKey bevat.
# Bij ontbrekende waarden faalt de validatie met een AssertionError.

assert sales_df2["ProductKey"].notna().all(), "ProductKey bevat ontbrekende waarde"
# Controleert of iedere rij een ProductKey bevat.
# Bij ontbrekende waarden faalt de validatie met een AssertionError.


assert (sales_df2["ReturnQuantity"] <= sales_df2["SalesQuantity"]).all(), "ReturnKey is bij één of meer regels groter dan SalesQuantity"
# Controleert of ReturnQuantity de SalesQuantity niet overschrijdt.
# Wanneer dit wel voorkomt, faalt de validatie met een AssertionError.


assert np.isclose(
    sales_df2["SalesAmountRet"],
    sales_df2["SalesAmount"] - sales_df2["ReturnAmount"],
    atol=0.01,
    rtol=0
).all() , "SalesAmountRet wijkt af van SalesAmount minus ReturnAmount."
# Controleert of SalesAmountRet gelijk is aan SalesAmount minus ReturnAmount.
# np.isclose wordt gebruikt om kleine afrondingsverschillen tot € 0,01 te accepteren.


assert np.isclose(
    sales_df2["TotalCostRet"],
    sales_df2["UnitCost"] *
    (sales_df2["SalesQuantity"] - sales_df2["ReturnQuantity"]),
    atol=0.01,
    rtol=0
).all(),  "TotalCostRet wijkt af van de herberekende kostprijs na retourcorrectie."
# Controleert of TotalCostRet overeenkomt met de herberekende kostprijs
# na correctie voor retouren.

# 2 ***Analyse product data***
In dit onderdeel wordt de product data set verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. Naar verwachting zijn er minimaal 1689 producten in de dataset.

## 2.1 Importeren dataset

In [60]:
#Importeren van Contoso product data
product_df = load_data("Contoso_Product.csv",
                       decimal=",")

## 2.2 Analyseren product dataset

In [61]:
#Functie aanhalen uit shared.py geschreven om te voorkoemn dat telkens dezelfde stappen doorlopen worden bij initiële analyse.
init_analyse(product_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   BrandName              1690 non-null   str  
 1   ClassName              1690 non-null   str  
 2   Manufacturer           1690 non-null   str  
 3   ProductDescription     1690 non-null   str  
 4   ProductKey             1690 non-null   int64
 5   ProductName            1690 non-null   str  
 6   ProductSubcategoryKey  1690 non-null   int64
 7   UnitCost               1690 non-null   str  
 8   UnitPrice              1690 non-null   str  
dtypes: int64(2), str(7)
memory usage: 374.9 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


BrandName                0
ClassName                0
Manufacturer             0
ProductDescription       0
ProductKey               0
ProductName              0
ProductSubcategoryKey    0
UnitCost                 0
UnitPrice                0
dtype: int64


Aantal unieke waarden:


BrandName                  10
ClassName                   3
Manufacturer               10
ProductDescription        516
ProductKey               1690
ProductName              1690
ProductSubcategoryKey      22
UnitCost                  397
UnitPrice                 350
dtype: int64


Eerste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
0,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",951,A. Datum Consumer Digital Camera E100 Black,23,"$143,26",$281
1,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",965,A. Datum Consumer Digital Camera E100 Grey,23,"$143,26",$281
2,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",979,A. Datum Consumer Digital Camera E100 Pink,23,"$143,26",$281
3,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",993,A. Datum Consumer Digital Camera E100 Silver,23,"$143,26",$281
4,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1007,A. Datum Consumer Digital Camera E100 Orange,23,"$143,26",$281
5,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1021,A. Datum Consumer Digital Camera E100 Green,23,"$143,26",$281
6,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1035,A. Datum Consumer Digital Camera E100 Azure,23,"$143,26",$281
7,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",950,A. Datum Super-zoom Digital Camera X300 Black,23,"$96,08",$290
8,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",964,A. Datum Super-zoom Digital Camera X300 Grey,23,"$96,08",$290
9,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",978,A. Datum Super-zoom Digital Camera X300 Pink,23,"$96,08",$290



Laatste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
1670,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",626,WWI Projector 480p LCD12 White,19,"$116,75",$229
1671,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",637,WWI Projector 480p LCD12 Silver,19,"$116,75",$229
1672,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,106,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1673,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,107,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1674,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,108,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1675,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,109,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1676,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,110,WWI Stereo Bluetooth Headphones New Generation...,6,"$61,16","$132,99"
1677,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",614,WWI Projector 720p DLP56 Black,19,"$459,4",$999
1678,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",625,WWI Projector 720p DLP56 White,19,"$459,4",$999
1679,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",636,WWI Projector 720p DLP56 Silver,19,"$459,4",$999


Uit de eerste analyse blijkt dat er 1690 unique keys in de dataset zitten. Dit sluit aan op de eerdere verwachting dat er minimaal 1689 unieke keys aanwezig dienen te zijn vanuit de sales data. Er is ook geen sprake van duplicaten op basis van deze analyse. 

Uit de weergave blijkt dat producten in verschillende kleuren beschikbaar zijn. De kleur zit op het moment in de ProductName ingesloten. Het lijkt beter om de naam en kleur te scheiden voor diepgaandere analyses op kleur en een zuiver analyses van bijvoorbeeld omzet of kosten behorend bij één product ongeacht van kleur.

Gekeken naar de datatypes zou de verwachting zijn dat de UnitCost en Unitprice als floats zouden worden ingeladen. Dit lijkt voorkomen te worden door de dollartekens in de cellen. Dit dit dient opgeschoond te worden. 

In de hele product dataset komen geen lege waarden voor.

De eerste stap is om de kosten en prijs om te zetten naar decimale waarden. Hiermee kan gecontroleerd worden of de kosten en prijs in de product dataset gelijk zijn aan die in de sales dataset.

In [62]:
#Verwijderen van de $ prefixes uit de kolommen UnitCost en UnitPrice. Daarna de komma vervangen om floats te kunnen maken.
product_df["UnitCost"] = product_df["UnitCost"].str.removeprefix("$").str.replace(",",".")
product_df["UnitPrice"] = product_df["UnitPrice"].str.removeprefix("$").str.replace(",",".")

In [63]:
#UnitCost en UnitPrice omzetten in floats.
product_df[["UnitCost","UnitPrice"]] = product_df[["UnitCost","UnitPrice"]].apply(pd.to_numeric)
product_df.loc[~product_df["ProductKey"].isin(sales_df2["ProductKey"])]
#Zoeken naar producten wel in product dataset maar niet in de salesdata set

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
35,A. Datum,Economy,A. Datum Corporation,"Digital camera, 3.7 in x 0.9 in x 2",944,A. Datum Compact Digital Camera M200 Black,23,59.32,129.00


In [64]:
#Zoeken naar producten wel in product dataset maar niet in de salesdata set
product_df.loc[~product_df["ProductKey"].isin(sales_df2["ProductKey"])]

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
35,A. Datum,Economy,A. Datum Corporation,"Digital camera, 3.7 in x 0.9 in x 2",944,A. Datum Compact Digital Camera M200 Black,23,59.32,129.00


De A. Datum Compact Digital Camera M200 Black zit wel in de product dataset maar wordt volgens de sales data niet verkocht in de periode waar deze betrekking op heeft

In [65]:
#Onderzoeken van strings in datset.
display(product_df["BrandName"].value_counts())
#Onderzoeken producten van ieder merk
display(product_df["ClassName"].value_counts())
#Onderzoeken segmenten
display(product_df["Manufacturer"].value_counts())
#Onderzoeken Producenten
display(product_df["ProductDescription"].value_counts())
#Onderzoeken productomschrijvingen
display(product_df["ProductName"].value_counts())
#Onderzoeken productnamen

BrandName
Contoso                 560
Proseware               177
Southridge Video        170
Fabrikam                163
The Phone Company       152
Wide World Importers    133
A. Datum                132
Adventure Works         128
Litware                  48
Northwind Traders        27
Name: count, dtype: int64

ClassName
Regular    930
Economy    516
Deluxe     244
Name: count, dtype: int64

Manufacturer
Contoso, Ltd            560
Proseware, Inc.         177
Southridge Video        170
Fabrikam, Inc.          163
The Phone Company       152
Wide World Importers    133
A. Datum Corporation    132
Adventure Works         128
Litware, Inc.            48
Northwind Traders        27
Name: count, dtype: int64

ProductDescription
1080i/p or 720p high definition or standard definition                                             35
6-hour rechargeable battery, 180-degree swivel and flip screen                                     30
Store your CDs or DVDs in these beautiful binders, designed to look like books                     24
Digital camera - SLR with live view mode                                                           21
720p high definition or VGA                                                                        20
                                                                                                   ..
16:9 Aspect Ratio - 113in.Dual wall and ceiling - perfect for home / business.                      1
16:9 Aspect Ratio - 125in. 160 degree wide viewing angle.                                           1
16:9 aspect ratio-113in, dual wall and ceiling - perfect for home / business                        1
2.3GHz dual processor, 4 GB DDR2 SDRAM memories, 4MB L2 Cache  

ProductName
A. Datum Consumer Digital Camera E100 Black             1
A. Datum Consumer Digital Camera E100 Grey              1
A. Datum Consumer Digital Camera E100 Pink              1
A. Datum Consumer Digital Camera E100 Silver            1
A. Datum Consumer Digital Camera E100 Orange            1
                                                       ..
WWI LCD20 M220 White                                    1
WWI Wireless Bluetooth Stereo Headphones M170 Silver    1
WWI Wireless Bluetooth Stereo Headphones M170 Black     1
WWI Wireless Bluetooth Stereo Headphones M170 White     1
WWI Wireless Bluetooth Stereo Headphones M170 Pink      1
Name: count, Length: 1690, dtype: int64

De verschillende merken, segmenten, producenten, productomschrijvingen en productnamen zijn geanalyseerd.

Opvallend is het feit dat er **516** unieke omschrijvingen zijn van producten, maar **1690** unieke productbenamingen.
Dit wordt met name veroorzaakt door het feit dat iedere product meerdere kleurvarianten heeft.   

Het lijkt aan te raden om een scheiding te maken tussen productnamen en kleuren. 
Zo kan er zuiver gekeken worden naar productverkopen en later naar de voorkeur van de kleur. 
De verwachting is dat na opschonen het aantal unieke productnamen zonder kleur gelijk is aan de unieke productomschrijvingen.

De volgende stap is om te kijken of product kosten en prijzen overeenkomen met de gegevens uit de sales set.

In [66]:
#Controleren of ProductUnitCost overeenkomt met gegevens in de sales dataset.

#De Unitcost van de productdataset koppelen aan de sales_df2 dataset.
sales_df2["ProductUnitCost"] = sales_df2["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitCost"]
)

#Controleren of ProductUnitCost in product gelijk is aan Unitcost in de sales dataset.
sales_df2["CostControl"] = (
    sales_df2["ProductUnitCost"] == sales_df2["UnitCost"]
)

#Controleren waar UnitCost niet overeenkomt.
display(sales_df2[~sales_df2["CostControl"]])


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,TotalCostRet,ProductUnitCost,CostControl


ProductUnitCost in de product dataset en UnitCost in de salesdataset zijn voor alle records gelijk.

In [67]:
sales_df2["ProductUnitPrice"] = sales_df2["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitPrice"]
)
#De UnitPrice van de productdataset koppelen aan de sales_df2 dataset.
sales_df2["PriceControl"] = (
    sales_df2["ProductUnitPrice"] == sales_df2["UnitPrice"]
)
#Controleren of UnitPrice gelijk is aan ProductUnitPrice voor alle records in sales data.
display(sales_df2[~sales_df2["PriceControl"]])


,SalesKey,DateKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,...,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,SalesAmountRet,TotalCostRet,ProductUnitCost,CostControl,ProductUnitPrice,PriceControl


In [68]:
#PriceControl en CostControl verwijderen uit de dataframe, omdat deze niet nodig is voor de definitieve dataset.
sales_df2 = sales_df2.drop(columns=["PriceControl", "CostControl"])

ProductUnitCost in de product dataset en UnitCost in de salesdataset zijn voor alle records gelijk.

Uit vergelijkingen met de sales dataset blijkt dat kosten en prijzen voor alle sales overeenkomen met die uit de productdata.

### **Tussentijdse bevindingen**

Conform eerdere verwachtingen bevat de product dataset minimaal 1689 productkeys, het totaal is 1690. 
Het enige product dat niet voorkomt in de sales dataset is de *A. Datum Comapct Digital Camera M200 black*.  
Verder is er gekeken naar volgende eigenschappen:
- Merknamen: 10 
- Segmenten: 3 
- Producten: 10 
- Unieke productomschrijvingen: 516 
- Unieke productnamen: 1690 


Het is opvallend dat er 516 productomschrijvingen zijn, tegenover 1690 productnamen. Dit lijkt veroorzaakt te worden doordat kleurnamen worden opgenomen in de ProductName, waarbij de omschrijving van het product hetzelfde blijft.

*To-do*:
- Aparte kolommen toevoegen voor productnaam en kleur.

## 2.3 Data Cleaning

### Scheiden van ProductName kolom

In [69]:
#Product kleuren opslaan naar ProductKolom.
product_df["ProductColor"] = product_df["ProductName"].str.split().str[-1]
#Controleren op waarden en aantallen.
product_df["ProductColor"].value_counts()


ProductColor
Black     485
White     356
Silver    261
Grey      195
Blue       70
Brown      64
Pink       56
Gold       47
Orange     42
Green      40
Red        39
Yellow     15
Azure      14
Purple      3
blue        2
E120        1
Name: count, dtype: int64

Eerst conversie laat zien dat dat er 2 verschillende schrijfwijze zijn voor blue. Eén met en één zonder hoofdletter. Daarnaast is er één product waarbij het laatste woord geen kleur is, waardoor de waarde E120 wordt teruggegeven.

In [70]:
#Schrijfwijze veranderen naar title, eerst letter is een hoofdletter, de rest kleine letters.
product_df["ProductColor"]=product_df["ProductColor"].str.title()
#Opnieuw tellen. Enkel het probleem met de E120 blijft.
display(product_df["ProductColor"].value_counts())
#Er staat geen kleur  bij deze waarde, dus de standaard splits wijze gaat niet op.
product_df.loc[product_df["ProductColor"]=="E120"]

ProductColor
Black     485
White     356
Silver    261
Grey      195
Blue       72
Brown      64
Pink       56
Gold       47
Orange     42
Green      40
Red        39
Yellow     15
Azure      14
Purple      3
E120        1
Name: count, dtype: int64

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor
421,Contoso,Economy,"Contoso, Ltd",Expect better quality than any other screen pr...,2494,Reusable Phone Screen Protector E120,33,1.50,2.94,E120


In [71]:
#Waarde zoeken in color kolom en vervangen door "No Color", zodat later bekend is dat deze geen kleur heeft.
product_df.loc[(product_df["ProductColor"]=="E120"),"ProductColor"]="No Color"

display(product_df["ProductColor"].value_counts())
display(product_df["ProductColor"].value_counts().sum())

ProductColor
Black       485
White       356
Silver      261
Grey        195
Blue         72
Brown        64
Pink         56
Gold         47
Orange       42
Green        40
Red          39
Yellow       15
Azure        14
Purple        3
No Color      1
Name: count, dtype: int64

np.int64(1690)

Alle productregels hebben nu een eigen kleur kolom. 1 product heeft geen kleur.
Nu dienen de producten nog gesplitst te worden.

In [72]:
#Splitsen productnamen van kleuren, zodat er één productnaam overblijft, op basis van de aanname dat het laatste woord de kleur betreft.
#split maakt een lijst van woorden, het laatste woord wordt verwijderd, de string wordt weer samengevoegd en zou de productnaam zonder kleur moeten weergeven.
product_df["ProductNameNoColor"] = product_df["ProductName"].str.split().str[:-1].str.join(" ")

#Controleren of laatste waarde van de productnaam zonder kleur in de kleur kolom zit.
product_df[product_df["ProductNameNoColor"].str.split().str[-1].isin(product_df["ProductColor"])]

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor,ProductNameNoColor
28,A. Datum,Economy,A. Datum Corporation,"Digital camera - SLR with live view mode, 5 in...",1058,A. Datum SLR Camera M138 Silver Grey,24,143.48,312.00,Grey,A. Datum SLR Camera M138 Silver
44,A. Datum,Regular,A. Datum Corporation,"5 in x 2.4 in x 3.9 in, 15.9 oz, digital camer...",1059,"A. Datum SLR Camera 35"" M358 Silver Grey",24,155.43,338.00,Grey,"A. Datum SLR Camera 35"" M358 Silver"
56,A. Datum,Regular,A. Datum Corporation,"Digital camera – compact, 3.8 in x 1 in x 2.2 ...",1042,A. Datum Point Shoot Digital Camera M500 Silve...,23,91.05,198.00,Grey,A. Datum Point Shoot Digital Camera M500 Silver
111,A. Datum,Regular,A. Datum Corporation,Dynamic integrated dust reduction system,1056,A. Datum SLR Camera X138 Silver Grey,24,207.74,627.00,Grey,A. Datum SLR Camera X138 Silver
113,A. Datum,Regular,A. Datum Corporation,Fast startup and split-second shutter response...,1057,"A. Datum SLR Camera 35"" X358 Silver Grey",24,194.82,588.00,Grey,"A. Datum SLR Camera 35"" X358 Silver"
122,A. Datum,Regular,A. Datum Corporation,"Optical zoom 2.7in LCD digital camera, 100kgf",1043,A. Datum Full Frame Digital Camera X300 Silver...,23,76.53,231.00,Grey,A. Datum Full Frame Digital Camera X300 Silver
297,Contoso,Deluxe,"Contoso, Ltd",Dynamic integrated dust reduction system,1088,Contoso SLR Camera X144 Silver Grey,24,222.98,673.00,Grey,Contoso SLR Camera X144 Silver
638,Contoso,Regular,"Contoso, Ltd",5 in x 2.4 in x 3.9 in. 15.9 oz. Digital camer...,1091,"Contoso SLR Camera 35"" M358 Silver Grey",24,164.63,358.00,Grey,"Contoso SLR Camera 35"" M358 Silver"
708,Contoso,Regular,"Contoso, Ltd","Digital camera - SLR with live view mode, 5 in...",1090,Contoso SLR Camera M144 Silver Grey,24,152.68,332.00,Grey,Contoso SLR Camera M144 Silver
735,Contoso,Regular,"Contoso, Ltd",Fast startup and split-second shutter response...,1089,"Contoso SLR Camera 35"" X358 Silver Grey",24,188.19,568.00,Grey,"Contoso SLR Camera 35"" X358 Silver"


Door de foutieve aanname dat alle kleuren 1 woord zijn is silver Grey nu niet goed in de kleur kolom gezet. Dit dient gecorrigeerd te worden alvorens verder te gaan. 

In [73]:
#Zoek de productnamen met "Silver Grey" en zet de waarde voor ProductColor naar Silver Grey.
product_df.loc[product_df["ProductName"].str.contains("Silver Grey"),"ProductColor"] = "Silver Grey"

In [74]:
#maak een list van kleuren.
kleuren = list(product_df["ProductColor"].unique())
#Masker dat controleert of kleur in de productnaam voor komt.
masker = product_df["ProductName"].apply(lambda naam: any(kleur in naam for kleur in kleuren))
#Laat de producten zien waarvoor dit niet van toepassing is. Dit zijn de producten die we eerder gecorrigeerd hebben.
display(product_df[~masker])

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor,ProductNameNoColor
421,Contoso,Economy,"Contoso, Ltd",Expect better quality than any other screen pr...,2494,Reusable Phone Screen Protector E120,33,1.50,2.94,No Color,Reusable Phone Screen Protector
444,Contoso,Economy,"Contoso, Ltd","Illuminate wireless reading device, two super ...",843,Contoso Bright Light battery E20 blue,22,13.71,26.90,Blue,Contoso Bright Light battery E20
599,Contoso,Regular,"Contoso, Ltd","2.4'' LCD Touch screen, 8GB flash memory, play...",44,Contoso 16GB New Generation MP5 Player M1650 blue,1,106.69,232.00,Blue,Contoso 16GB New Generation MP5 Player M1650


Kleur lijkt nu opgelost te zijn, we zien alleen false positives voor No Color en producten waarbij blue zonder hoofdletter is geschreven.

In [75]:
#Nieuwe functie die met behulp van apply voor iedere rij een nieuwe waarde berekent die de productcolor verwijdert van de originele productname.
product_df["ProductNameNoColor"] = product_df.apply(
    lambda row: row["ProductName"].removesuffix(row["ProductColor"]).strip(),
    axis=1
)

In [76]:
product_df["ProductNameNoColor"] = product_df.apply(
    # Voert de onderstaande lambda-functie uit op iedere rij van product_df.
    lambda row: re.sub(
        # Vervangt tekst op basis van een regex-patroon.
        rf"{re.escape(row['ProductColor'])}$",
        # Bouwt het zoekpatroon op uit de kleur van de huidige rij.
        # re.escape() zorgt dat eventuele speciale regextekens in de kleur veilig worden behandeld.
        # $ betekent: de kleur moet aan het einde van ProductName staan.
        "",
        # Vervangt de gevonden kleur door een lege string.
        row["ProductName"],
        # Dit is de tekst waarin gezocht en vervangen wordt.
        flags=re.IGNORECASE
        # Negeert hoofdletters en kleine letters.
        # Dus "Blue", "blue" en "BLUE" worden allemaal gevonden.
    ).strip(),
    # Verwijdert eventuele overgebleven spaties aan het begin of einde.
    axis=1
    # Geeft aan dat apply rij voor rij moet werken in plaats van kolom voor kolom.
)

In [77]:
#Dataframe maken met alleen productnamen en descriptions.
Duplicatecontrole = product_df[["ProductNameNoColor","ProductDescription"]].drop_duplicates()
#Aantal keren tellen dat waarden voorkomen. 
Duplicatecontrole["ProductDescription"].value_counts()

ProductDescription
1080i/p or 720p high definition or standard definition                                                        20
Digital camera - SLR with live view mode                                                                      12
6-hour rechargeable battery, 180-degree swivel and flip screen                                                10
DVD and VHS recorder with two way dubbing, records -R/-RW, +R/+RW formats                                     10
720p high definition or VGA                                                                                    9
                                                                                                              ..
2.3GHz dual processor, 4 GB DDR2 SDRAM memories, 4MB L2 Cache                                                  1
2GB flash memory, USB driver, continuous recording until memory full, as spy cameras, hidden cameras           1
3.0GHz processor, 4 GB DDR2 SDRAM memories, 500GB 7200RPM hard drive, 19 inch

De dataframe met alleen productnames en descriptions geeft 516 unieke descriptions met maximaal 20 keer hergebruikt en minimaal 1 keer.
Dit bevestigt dus dat er voor meerdere producten dezelfde description wordt gebruikt.  

In [78]:
#Masker maken ter controle van de kleuren in de dataset.
masker_color = pd.Series(
    [
        pd.notna(kleur)
        #Controleert of waarden in ProductColor gevuld is.
        and pd.notna(naam)
        # Controleert of ProductNameNoColor gevuld is.
        and bool(
            re.search(
                rf"\b{re.escape(str(kleur))}\b",
                str(naam),
                flags=re.IGNORECASE
            )
        )
        #Controleert of ProductColor in ProductNameNoColor voorkomt. De regex-expressie met \b voorkomt dat Blue in Bluetooth een false positive geeft.
        for naam, kleur in zip(
            product_df["ProductNameNoColor"],
            product_df["ProductColor"]
        )
        #Geeft aan voor welke waarden dit gecontroleerd moet worden. 
        #Zip vormt een iterator die per iteratie een combinatie van ProductNameNoColor en ProductColor uit dezelfde rij oplevert.
    ],
    index=product_df.index
    #Index van dataframe toepassen op masker.
)

product_df[masker_color]
#Laat rijen zien, waarvoor de kleur nog wel in ProductNameNoColor voorkomt.

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice,ProductColor,ProductNameNoColor


## 2.4 Validatie

In [79]:
assert product_df["ProductKey"].is_unique,  "ProductKey bevat één of meerdere duplicaten."
#Controleert of alle ProductKeys uniek zijn.

assert product_df["ProductKey"].notna().all(), "ProductKey bevat één of meerdere ontbrekende waarden."
#Controleert of er waarden ontbreken in de ProductKey.

assert not masker_color.any(), "Eén of meerdere productkleuren worden nog benoemd in de productnaam zonder kleur."
#Controleert of er kleuren voorkomen in de ProductNoColor kolom.

# 3 ***Analyse calendar data***
In dit onderdeel kijken we naar de calendar dataset.
Naar verwachting zal deze matchen op de datekey en minimaal datums van 1 januari 2011 tot en met 31 december 2013 bevatten.

### 3.1 Importeren dataset
Met behulp van path en pandas importeren wordt de dataframe geïmporteerd voor analyse.

In [80]:
#Inlezen calendar data set.
calendar_df = load_data("Contoso_Calendar.csv",
                       decimal=",")

### 3.2 Analyse dataset
Hier voeren we de eerste analyse uit op de calendar dataset.

In [81]:
#Met behulp van analyse functie eerste algemene gegeens analyseren.
init_analyse(calendar_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 5113 entries, 0 to 5112
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   DateInt        5113 non-null   int64
 1   Year           5113 non-null   int64
 2   Quarter        5113 non-null   str  
 3   Month          5113 non-null   str  
 4   Day            5113 non-null   int64
 5   DayOfMonth     5113 non-null   int64
 6   DayOfWeekName  5113 non-null   str  
 7   MonthName      5113 non-null   str  
 8   MonthOfYear    5113 non-null   int64
 9   QuarterOfYear  5113 non-null   int64
 10  Year.1         5113 non-null   int64
dtypes: int64(7), str(4)
memory usage: 561.6 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


DateInt          0
Year             0
Quarter          0
Month            0
Day              0
DayOfMonth       0
DayOfWeekName    0
MonthName        0
MonthOfYear      0
QuarterOfYear    0
Year.1           0
dtype: int64


Aantal unieke waarden:


DateInt          5113
Year               14
Quarter             4
Month              12
Day                31
DayOfMonth         31
DayOfWeekName       7
MonthName          12
MonthOfYear        12
QuarterOfYear       4
Year.1             14
dtype: int64


Eerste 20 rijen:


,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1
0,20050101,2005,Qtr 1,January,1,1,Saturday,January,1,1,2005
1,20050102,2005,Qtr 1,January,2,2,Sunday,January,1,1,2005
2,20050103,2005,Qtr 1,January,3,3,Monday,January,1,1,2005
3,20050104,2005,Qtr 1,January,4,4,Tuesday,January,1,1,2005
4,20050105,2005,Qtr 1,January,5,5,Wednesday,January,1,1,2005
5,20050106,2005,Qtr 1,January,6,6,Thursday,January,1,1,2005
6,20050107,2005,Qtr 1,January,7,7,Friday,January,1,1,2005
7,20050108,2005,Qtr 1,January,8,8,Saturday,January,1,1,2005
8,20050109,2005,Qtr 1,January,9,9,Sunday,January,1,1,2005
9,20050110,2005,Qtr 1,January,10,10,Monday,January,1,1,2005



Laatste 20 rijen:


,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1
5093,20181212,2018,Qtr 4,December,12,12,Wednesday,December,12,4,2018
5094,20181213,2018,Qtr 4,December,13,13,Thursday,December,12,4,2018
5095,20181214,2018,Qtr 4,December,14,14,Friday,December,12,4,2018
5096,20181215,2018,Qtr 4,December,15,15,Saturday,December,12,4,2018
5097,20181216,2018,Qtr 4,December,16,16,Sunday,December,12,4,2018
5098,20181217,2018,Qtr 4,December,17,17,Monday,December,12,4,2018
5099,20181218,2018,Qtr 4,December,18,18,Tuesday,December,12,4,2018
5100,20181219,2018,Qtr 4,December,19,19,Wednesday,December,12,4,2018
5101,20181220,2018,Qtr 4,December,20,20,Thursday,December,12,4,2018
5102,20181221,2018,Qtr 4,December,21,21,Friday,December,12,4,2018


De calendar dataset is groter dan de sales dataset qua datumbereik. Dit is opzich geen probleem, echter kan dit wel voor extra load zorgen gezien gegevens die niet worden gebruikt worden ingeladen.
Een actiepunt is om eventueel later te kijken naar het schrappen van datums die niet relevant zijn om performance te verbeteren.
Alle waarden zijn gevuld en volgen een standaard structuur. 
De Year kolom lijkt 2 keer aanwezig, eventueel kan er één gedropt worden uit de set. Daarnaast lijken Day en DayOfMonth dezelfde gegevens te weergeven. 

Op het moment is het niet zeker of deze tabel in PowerBI direct werkt, voor pandas gebruik hebben we een kolom nodig die als datetime functioneert. 
De volgende stap is om op voorhand een kolom te maken die als datetime gebruikt kan worden. 

In [82]:
#Controleren matches in sales dataset.
calendar_df["DateInt"].isin(sales_df["DateKey"])

0       False
1       False
2       False
3       False
4       False
        ...  
5108    False
5109    False
5110    False
5111    False
5112    False
Name: DateInt, Length: 5113, dtype: bool

In de sales data wordt gebruik gemaakt van een DateKey kolom. Deze waarde komt niet terug in de calendar dataset en matcht ook niet op DateInt.
De DateInt dient nog als Key aangemaakt te worden als waarde in de sales set.

### 3.3 Data cleaning
In dit onderdeel maken we met de aanwezige gegevens een DateKey om te matchen op de waarde in de sales set.

In [83]:
#Datekey aanmaken in calendar set door de DateInt te converteren naar een datetime op basis van het format.
calendar_df["DateKey"] = pd.to_datetime(calendar_df["DateInt"], format="%Y%m%d")


C:\Users\x3ri0\AppData\Local\Temp\ipykernel_6624\3158511271.py:2: Pandas4Warning: Parsing integer or float values with a format in to_datetime is deprecated. In a future version these will be interpreted as epochs via the 'unit' keyword instead. Cast them to strings first to retain the current behavior.
  calendar_df["DateKey"] = pd.to_datetime(calendar_df["DateInt"], format="%Y%m%d")


In [84]:
#Controleren of de DateKey matcht op waarden in de sales data. 
calendar_df[calendar_df["DateKey"].isin(sales_df2["DateKey"])]

,DateInt,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1,DateKey
2191,20110101,2011,Qtr 1,January,1,1,Saturday,January,1,1,2011,2011-01-01
2192,20110102,2011,Qtr 1,January,2,2,Sunday,January,1,1,2011,2011-01-02
2193,20110103,2011,Qtr 1,January,3,3,Monday,January,1,1,2011,2011-01-03
2194,20110104,2011,Qtr 1,January,4,4,Tuesday,January,1,1,2011,2011-01-04
2195,20110105,2011,Qtr 1,January,5,5,Wednesday,January,1,1,2011,2011-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...
3282,20131227,2013,Qtr 4,December,27,27,Friday,December,12,4,2013,2013-12-27
3283,20131228,2013,Qtr 4,December,28,28,Saturday,December,12,4,2013,2013-12-28
3284,20131229,2013,Qtr 4,December,29,29,Sunday,December,12,4,2013,2013-12-29
3285,20131230,2013,Qtr 4,December,30,30,Monday,December,12,4,2013,2013-12-30


Met behulp van de DateIn waarde is een DateKey aangemaakt die matcht op gegevens in de sales dataset. 
Bij Controle van de dataset start deze op 1 januari 2011 en eindigt deze op 31 december 2013. Dit komt overeen met de sales dataset.
Dit lijkt voornamelijk relevant voor het PowerBI gedeelte.

In [85]:
#Waarden van Datekey veranderen naar integer door middel van strftime en astype. Dit is nodig om de DateKey te kunnen matchen met de DateInt in de calendar dataset.
sales_df2["DateInt"] = sales_df2["DateKey"].dt.strftime("%Y%m%d").astype(int)

In [86]:
#Controleren of er waarden zijn die niet voorkomen in calendar.
sales_df2["DateInt"].isin(calendar_df["DateInt"]).value_counts()

DateInt
True    2282482
Name: count, dtype: int64

In [87]:
#Columns renamen, zodat key koppelt op DateKey in calendar.
sales_df2 = sales_df2.rename(columns={"DateKey":"Date",
                          "DateInt":"DateKey"})

calendar_df = calendar_df.rename(columns={"DateKey":"Date"})

calendar_df = calendar_df.rename(columns={"DateInt":"DateKey"})

display(calendar_df)

,DateKey,Year,Quarter,Month,Day,DayOfMonth,DayOfWeekName,MonthName,MonthOfYear,QuarterOfYear,Year.1,Date
0,20050101,2005,Qtr 1,January,1,1,Saturday,January,1,1,2005,2005-01-01
1,20050102,2005,Qtr 1,January,2,2,Sunday,January,1,1,2005,2005-01-02
2,20050103,2005,Qtr 1,January,3,3,Monday,January,1,1,2005,2005-01-03
3,20050104,2005,Qtr 1,January,4,4,Tuesday,January,1,1,2005,2005-01-04
4,20050105,2005,Qtr 1,January,5,5,Wednesday,January,1,1,2005,2005-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...
5108,20181227,2018,Qtr 4,December,27,27,Thursday,December,12,4,2018,2018-12-27
5109,20181228,2018,Qtr 4,December,28,28,Friday,December,12,4,2018,2018-12-28
5110,20181229,2018,Qtr 4,December,29,29,Saturday,December,12,4,2018,2018-12-29
5111,20181230,2018,Qtr 4,December,30,30,Sunday,December,12,4,2018,2018-12-30


#### **Voorlopige conclusie**

De calendar data lijkt met name voor de tijdfunctie binnen PowerBI.
Voor de zekerheid is een key aangemaakt om op te matchen met de sales data.  
Day en DayOfMonth lijken dezelfde waarde te bevatten, idem voor Year en Year.1

*actiepunten*:
- Eventueel dubbele waarden voor day en year laten vallen.
- Tabel verkleinen tot bereik van de dataset.

In [88]:
calendar_df = calendar_df.drop(["Day","Year.1"],axis=1)

Dubbele waarden zijn verwijderd en er is een werkende datekey gemaakt.

In [89]:
calendar_df2 = calendar_df[calendar_df["Date"].between("01-01-2011","31-12-2013")].reset_index(drop=True)
#Een nieuwe calendar dataset met alleen datums relevant tot de dataset om performance optimaal te houden.

Voor de dataset in PowerBI is een nieuwe calendar dataframe aangemaakt die alleen datums bevat die binnen het bereik van de salesdataset vallen.

## 3.4 Validatie

In [90]:
assert calendar_df2["DateKey"].is_unique, "DateKey bevat één of meerdere dubbele datums."
#Controleert of alle DateKeys uniek zijn.

assert calendar_df2["DateKey"].notna().all(), "DateKey bevat één of meerdere ontbrekende waarden."
#Controleert of er lege waarden in de DateKey aanwezig zijn.

assert pd.api.types.is_datetime64_any_dtype(calendar_df2["Date"]), "DateKey is niet omgezet naar een geldig datetime datatype."
#Controleert of DateKey correct is omgezet naar een datumwaarde.

# 4 ***Analyse geography data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. 

### 4.1 Importeren dataset


In [91]:
#Inlezen geography dataset
geography_df = load_data("Contoso_Geography.csv",
                       decimal=",")

### 4.2 Analyse dataset

In [92]:
#Initiële analyse met behulp van init_analyse functie
init_analyse(geography_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 674 entries, 0 to 673
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ContinentName      674 non-null    str  
 1   GeographyKey       674 non-null    int64
 2   GeographyType      674 non-null    str  
 3   RegionCountryName  671 non-null    str  
dtypes: int64(1), str(3)
memory usage: 38.2 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ContinentName        0
GeographyKey         0
GeographyType        0
RegionCountryName    3
dtype: int64


Aantal unieke waarden:


ContinentName          3
GeographyKey         674
GeographyType          4
RegionCountryName     34
dtype: int64


Eerste 20 rijen:


,ContinentName,GeographyKey,GeographyType,RegionCountryName
0,Asia,1,Continent,NaN
1,Asia,269,Country/Region,Armenia
2,Asia,270,Country/Region,Australia
3,Asia,271,Country/Region,Bhutan
4,Asia,272,Country/Region,China
5,Asia,273,Country/Region,India
6,Asia,274,Country/Region,Iran
7,Asia,275,Country/Region,Japan
8,Asia,276,Country/Region,Kyrgyzstan
9,Asia,277,Country/Region,Pakistan



Laatste 20 rijen:


,ContinentName,GeographyKey,GeographyType,RegionCountryName
654,North America,933,City,United States
655,North America,934,City,United States
656,North America,935,City,United States
657,North America,936,City,United States
658,North America,937,City,United States
659,North America,938,City,United States
660,North America,939,City,United States
661,North America,940,City,United States
662,North America,941,City,United States
663,North America,942,City,Canada


In [93]:
#Verschillende geographytypes en aantallen bekijken
geography_df["GeographyType"].value_counts()

GeographyType
City              517
State/Province    120
Country/Region     34
Continent           3
Name: count, dtype: int64

De dataset lijkt opgebouwd uit continenten, landen, provincies/regios en steden.  
Bij regioncountryname zijn er 3 lege waarden gevonden in de initiële analyse. Hier dient op ingezoomd te worden. Er is geen sprake van duplicaten.

In [94]:
#Waarden met ontbrekende regioncountryname bekijken.
geography_df[geography_df["RegionCountryName"].isna()]

,ContinentName,GeographyKey,GeographyType,RegionCountryName
0,Asia,1,Continent,NaN
114,Europe,2,Continent,NaN
335,North America,3,Continent,NaN


De lege waarden lijken betrekking hebben op contintent.
Deze wordt ook al in ContinentName benoemd. 
Het lijkt dus logisch om deze waarden te verwijderen uit de dataset om de data duidelijke te presenteren.

In [95]:
#Onderzoeken dataset op afwijkende schrijfwijzes, daarnaast de samenstelling van de geography set.
geographystr = ["ContinentName","GeographyType","RegionCountryName"]

for g in geographystr:
    display(geography_df[g].value_counts())


ContinentName
North America    339
Europe           221
Asia             114
Name: count, dtype: int64

GeographyType
City              517
State/Province    120
Country/Region     34
Continent           3
Name: count, dtype: int64

RegionCountryName
United States      312
France              65
Germany             54
Australia           47
United Kingdom      47
Canada              26
Japan               13
China               11
Italy               11
India                7
Russia               7
South Korea          5
Armenia              3
Bhutan               3
Iran                 3
Kyrgyzstan           3
Pakistan             3
Singapore            3
Syria                3
Taiwan               3
Thailand             3
Turkmenistan         3
Denmark              3
Greece               3
the Netherlands      3
Ireland              3
Malta                3
Poland               3
Portugal             3
Romania              3
Slovenia             3
Spain                3
Sweden               3
Switzerland          3
Name: count, dtype: int64

In [96]:
# GeographyKey moet uniek zijn
display(geography_df["GeographyKey"].is_unique)

# Verdeling over hiërarchieniveaus
display(geography_df["GeographyType"].value_counts())

# Zelfde naam op verschillende niveaus bekijken
display(geography_df[
    geography_df["RegionCountryName"].eq("the Netherlands")
])

True

GeographyType
City              517
State/Province    120
Country/Region     34
Continent           3
Name: count, dtype: int64

,ContinentName,GeographyKey,GeographyType,RegionCountryName
119,Europe,288,Country/Region,the Netherlands
151,Europe,349,State/Province,the Netherlands
290,Europe,736,City,the Netherlands


GeographyKeys lijken zowel betrekking te kunnen hebben op meerdere GeographyTypes.  
Hierdoor heeft ieder land minimaal 3 weergaven.  
 Bij meerdere cities of provinces kunnen dit er meer dan 3 zijn.

## 4.3 Data Cleaning

In [97]:
#Nieuwe dataframe maken waarin continents uit geographytype worden gedropt voor duidelijkere analyses.
geography_df2 = geography_df.loc[~(geography_df["RegionCountryName"].isna())]

Contintent gedropt uit de data set, gezien deze ook al benoemd wordt in de kolom "ContinentName".

## 4.4 Validatie

In [98]:
assert geography_df2["GeographyKey"].is_unique, "GeographyKey bevat één of meerdere duplicaten."
# Controleert of alle GeographyKeys uniek zijn.


assert geography_df2["GeographyKey"].notna().all(),  "GeographyKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere rij een GeographyKey bevat.


assert geography_df2["RegionCountryName"].notna().all(), "RegionCountryName bevat één of meerdere ontbrekende waarden."
# Controleert of de continentregels succesvol zijn verwijderd en iedere overgebleven rij een RegionCountryName bevat.


assert geography_df2["ContinentName"].notna().all(), "ContinentName bevat één of meerdere ontbrekende waarden."
# Controleert of ContinentName in alle kolommen is ingevuld.

#### Voorlopige conclusie
De geography dataset bevat gegevens omtrent locaties. Deze kunnen niet direct gelinkt worden aan de sales dataset.  
De voorlopige aanname is dat deze samenhangt met store data.
De gegevens hebben met name betrekking op de Verenigde Staten, daarop volgt Europa en uiteindelijk Europa.

Het zou interessant kunnen zijn om ontwikkelingen van de verkopen op deze verschillende contintenten te zien en vergelijken. 
Eventueel product context.

# 5 ***Analyse store data***
In dit onderdeel wordt de store dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales dataset en de geography dataset

### 5.1 Data importeren

In [99]:
#Inlezen stores.csv
stores_df = load_data("Contoso_Stores.csv",
                       decimal=",")

### 5.2 Data analyseren

In [100]:
#Initiële analyse met behulp van de init_analyse functie
init_analyse(stores_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CloseReason      12 non-null     str    
 1   EmployeeCount    305 non-null    float64
 2   GeographyKey     306 non-null    int64  
 3   SellingAreaSize  306 non-null    int64  
 4   Status           306 non-null    str    
 5   StoreKey         306 non-null    int64  
 6   StoreName        306 non-null    str    
 7   StoreType        306 non-null    str    
dtypes: float64(1), int64(3), str(4)
memory usage: 28.5 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


CloseReason        294
EmployeeCount        1
GeographyKey         0
SellingAreaSize      0
Status               0
StoreKey             0
StoreName            0
StoreType            0
dtype: int64


Aantal unieke waarden:


CloseReason          2
EmployeeCount       46
GeographyKey       263
SellingAreaSize     15
Status               2
StoreKey           306
StoreName          306
StoreType            4
dtype: int64


Eerste 20 rijen:


,CloseReason,EmployeeCount,GeographyKey,SellingAreaSize,Status,StoreKey,StoreName,StoreType
0,NaN,19.00,424,455,On,4,Contoso Bellevue Store,Store
1,NaN,26.00,449,680,On,156,Contoso Cambridge Store,Store
2,NaN,19.00,453,455,On,88,Contoso Cedar Park Store,Store
3,NaN,25.00,501,680,On,214,Contoso Leeds Store,Store
4,NaN,17.00,506,462,On,201,Contoso London Store,Store
5,NaN,12.00,529,680,On,215,Contoso Paris Store,Store
6,NaN,12.00,529,551,On,309,Contoso Europe Reseller,Reseller
7,NaN,25.00,551,700,On,17,Contoso Spokane Store,Store
8,NaN,20.00,558,480,On,253,Contoso Sydney No.1 Store,Store
9,NaN,95.00,558,480,On,300,Contoso Sydney No.2 Store,Store



Laatste 20 rijen:


,CloseReason,EmployeeCount,GeographyKey,SellingAreaSize,Status,StoreKey,StoreName,StoreType
286,NaN,19.00,946,455,On,181,Contoso Virginia Beach Store,Store
287,NaN,25.00,947,700,On,10,Contoso Wapato Store,Store
288,NaN,33.00,948,560,On,174,Contoso Waterbury Store,Store
289,NaN,19.00,949,455,On,50,Contoso Waukesha No.1 Store,Store
290,NaN,17.00,949,462,On,54,Contoso Waukesha No.2 Store,Store
291,NaN,47.00,950,1125,On,21,Contoso Wheat Ridge Store,Store
292,NaN,33.00,951,560,On,182,Contoso Winchester Store,Store
293,NaN,25.00,952,700,On,147,Contoso Worcester No.1 Store,Store
294,Relocation,22.00,783,500,Off,184,Contoso Alexandria Store,Store
295,Relocation,22.00,851,500,Off,84,Contoso Humble Store,Store


Volgens de dataset zijn er 306 unieke storekeys dit komt overeen met eerdere analyse uit de sales data.  
Met name in de CloseReason kolom zijn veel lege waarden. De aanname is dat deze pas wordt ingevuld bij een sluiting van een locatie. Er is ook geen rekening gehouden met SCD. Een tabel die registreert wanneer de CloseReason wijzigt zou meer info kunnen geven over deze wijzigingen.  
Er is geen data aanwezig over wanneer de winkel is gesloten, het lijkt dus niet mogelijk te oordelen of er foutieve toewijzigingen aan een winkel zijn in de salessset.
Storetypes lijkt samen te kunnen hangen met de verkoopkanalen die voort komen uit de sales dataset.

Daarnaast ontbreekt er bij 1 store een employeecount, hier dient naar gekeken te worden. Tevens worden employeecounts op het moment als floats opgeslagen, ervanuit gaande dat dit om personen gaat en niet FTE.

Zoals eerder verwacht is de geographykey verwerkt in deze dataset.

In [101]:
#Record met ontbrekende employeecount bekijken.
display(stores_df[stores_df["EmployeeCount"].isna()])

,CloseReason,EmployeeCount,GeographyKey,SellingAreaSize,Status,StoreKey,StoreName,StoreType
72,NaN,NaN,743,680,On,210,Contoso Cheshire Store,Store


Betreft store met ontbrekende EmployeeCount. 

In [102]:
#Korte analyse op winkel
display(sales_df2.loc[(sales_df2["StoreKey"] == 210),["SalesKey","Date","SalesAmountRet"]].sort_values(by="Date").resample("QE", on="Date").agg(aantal_sales=("SalesKey","count"),
                                                                                                                                                         Omzet=("SalesAmountRet","sum")))

,aantal_sales,Omzet
Date,,
2011-03-31,744,"1,965,210.17"
2011-06-30,737,"2,111,682.17"
2011-09-30,731,"1,826,237.73"
2011-12-31,774,"1,613,786.97"
2012-03-31,320,"1,091,308.75"
2012-06-30,331,"1,120,883.17"
2012-09-30,327,"940,646.78"
2012-12-31,336,"754,083.41"
2013-03-31,234,"895,362.23"


Lijkt op een winkel met teruglopende omzet, wellicht daarom geen werknemers?

In [103]:
#Verdere analyse van winkel tegenover rest van dataset.
per_store_kwartaal = (
    sales_df2
    .groupby([
        "StoreKey",
        pd.Grouper(key="Date", freq="QE")
    ])
    .agg(
        aantal_sales=("SalesKey", "count"),
        omzet=("SalesAmountRet", "sum")
    )
)

per_store_kwartaal

gemiddelde_per_kwartaal = (
    per_store_kwartaal
    .groupby("Date")
    .agg(
        gemiddeld_aantal_sales=("aantal_sales", "mean"),
        mediaan_aantal_sales=("aantal_sales", "median"),
        gemiddelde_omzet=("omzet", "mean"),
        mediaan_omzet=("omzet", "median")
    )
)

gemiddelde_per_kwartaal

store_210 = per_store_kwartaal.loc[210]

vergelijking = store_210.join(
    gemiddelde_per_kwartaal
)

vergelijking

,aantal_sales,omzet,gemiddeld_aantal_sales,mediaan_aantal_sales,gemiddelde_omzet,mediaan_omzet
Date,,,,,,
2011-03-31,744,"1,965,210.17",723.18,524.00,"1,989,718.56","1,138,410.90"
2011-06-30,737,"2,111,682.17",923.97,770.00,"2,784,935.22","2,021,630.04"
2011-09-30,731,"1,826,237.73",742.09,524.50,"2,564,505.76","1,709,632.55"
2011-12-31,774,"1,613,786.97",842.39,640.50,"2,890,667.89","1,954,211.59"
2012-03-31,320,"1,091,308.75",564.07,352.00,"1,882,159.61","989,092.60"
2012-06-30,331,"1,120,883.17",598.49,365.50,"2,214,768.99","1,147,073.07"
2012-09-30,327,"940,646.78",595.78,360.00,"2,356,873.00","1,341,756.76"
2012-12-31,336,"754,083.41",596.18,363.00,"2,364,978.39","1,353,584.33"
2013-03-31,234,"895,362.23",470.34,245.00,"1,791,924.47","840,535.18"


Winkel blijft qua verkopen en omzet achter op gemiddelde en mediaan. Echter geen conclusie vast kunnen maken aan reden dat er geen employee count is ingegeven. 
Eventueel rij droppen als deze problemen geeft.

In [104]:
#Onderzoeken CloseReason
control = ["CloseReason","Status","StoreName","StoreType"]
display(stores_df.nunique())

#Loop om values te weergeven bij unieke waardes.
for c in control:
    display(stores_df[c].value_counts())




CloseReason          2
EmployeeCount       46
GeographyKey       263
SellingAreaSize     15
Status               2
StoreKey           306
StoreName          306
StoreType            4
dtype: int64

CloseReason
Relocation    10
Store          2
Name: count, dtype: int64

Status
On     294
Off     12
Name: count, dtype: int64

StoreName
Contoso Bellevue Store          1
Contoso Cambridge Store         1
Contoso Cedar Park Store        1
Contoso Leeds Store             1
Contoso London Store            1
                               ..
Contoso Stoughton Store         1
Contoso Trenton No.2 Store      1
Contoso Worcester No.2 Store    1
Contoso Buffalo Store           1
Contoso Cheney Store            1
Name: count, Length: 306, dtype: int64

StoreType
Store       299
Reseller      3
Online        3
Catalog       1
Name: count, dtype: int64

De volgende feiten vallen op gekeken naar de verschillende omschrijvingen en namen:
- 2 sluitingsreden
- 263 geographykeys t.o.v. 674 in de geography set
- 15 selling area sizes, de vraag is of dit daadwerkelijke oppervlakte is of een soort bucket?
- 4 storetypes, lijkt dus gelijk te zijn aan de verkoopkanalen.

 

In [105]:
#Controleren of alle waarden aanwezig zijn in salesset
stores_df["StoreKey"].isin(sales_df2["StoreKey"])
sales_df2["StoreKey"].isin(stores_df["StoreKey"])

0          True
1          True
2          True
3          True
4          True
           ... 
2282477    True
2282478    True
2282479    True
2282480    True
2282481    True
Name: StoreKey, Length: 2282482, dtype: bool

In [106]:
#GeographyKeys niet in de store dataset.
display(geography_df2[~geography_df2["GeographyKey"].isin(stores_df["GeographyKey"])])

#GeographyKeys in de store set.
display(geography_df2[(geography_df2["GeographyKey"].isin(stores_df["GeographyKey"]))])

#GeographyKeys in de store set, waarvan het GeographyType niet City is.
display(geography_df2.loc[(geography_df2["GeographyKey"].isin(stores_df["GeographyKey"]))&(geography_df2["GeographyType"]!="City")])


,ContinentName,GeographyKey,GeographyType,RegionCountryName
1,Asia,269,Country/Region,Armenia
2,Asia,270,Country/Region,Australia
3,Asia,271,Country/Region,Bhutan
4,Asia,272,Country/Region,China
5,Asia,273,Country/Region,India
...,...,...,...,...
498,North America,694,City,United States
499,North America,695,City,Canada
500,North America,701,City,Canada
501,North America,703,City,United States


,ContinentName,GeographyKey,GeographyType,RegionCountryName
65,Asia,558,City,Australia
86,Asia,708,City,Turkmenistan
87,Asia,709,City,Thailand
88,Asia,710,City,China
89,Asia,711,City,Kyrgyzstan
...,...,...,...,...
669,North America,948,City,United States
670,North America,949,City,United States
671,North America,950,City,United States
672,North America,951,City,United States


,ContinentName,GeographyKey,GeographyType,RegionCountryName


#### Voorlopige Conclusie
Storekeys in de salesset komen overeen met de sales set. 
Een groot aantal geography locaties komen niet voor in de storeset. 
Dit komt met name doordat er zowel waarden aanwezig zijn voor Country/region, province en city. Deze worden dus op meerdere lagen meegenomen.
Verdere analyse laat zien dat de storedataset alleen informatie op city niveau bevat. De Continentname en RegioCountryname kolommen geven hier invulling aan het continent en het land, over de provincie en city zijn geen verdere informatie in de dataset aanwezig. 

Er zijn geen historische gegevens beschikbaar met betrekking tot de kolom CloseReason. Daarnaast ontbreken definities van de waarden "Store" en "Relocation".

Wellicht interessant om in kaart te brengen of er locaties/regio's zijn waar veel verkopen zijn via andere kanalen waar een store van toegevoegde waarde zou zijn.
Het lijkt logischer de GeographyKey direct aan de sales te verbinden gezien dit een dimensie van de verkoop is

*To-do*:
- GeographyKey toevoegen aan sales voor directe link.


## 5.3 Data cleaning

### Geographykey toeveogen aan sales dataset

In [107]:
#Sales_df 2 kopieeren naar een nieuwe variabele te testen of de mapping werkt.
mergeid = sales_df2.copy()
#GeographyKey toevoegen door middel van  .map functie. Als de StoreKey in mergeid wordt gevonden, wordt de bijbehorende GeographyKey uit de storedataset toegevoegd.
mergeid["GeographyKey"] = mergeid["StoreKey"].map(stores_df.set_index("StoreKey")["GeographyKey"])
#Nieuwe sales data set met geographykey wordt leidend.
sales_df2 = mergeid
#Controleren of aantal geographykeys in sales overeenkomt met de eerdere controle, hoeveel geographykeys aanwezig zijn in de dataset.
display(geography_df2["GeographyKey"].isin(sales_df2["GeographyKey"]).value_counts())

GeographyKey
False    408
True     263
Name: count, dtype: int64

De GeographyKey is direct gelinkt aan de salesdata gezet, omdat dit een dimensie van sales is.  
Controle uitgevoerd door te controleren of aantal GeographyKeys uit de Geography dataset aansluit met de waarde die de vergelijking tussen stores en geography eerder gaf.  
Er is voor gekozen de GeographyKey ook in de storeset te behouden, zodat er ook nog specifieke data over winkels opgevraagd kan worden.

## 5.4 Validatie


In [108]:
assert stores_df["StoreKey"].is_unique,  "StoreKey bevat één of meerdere duplicaten."
#Controleert of er duplicaten in de StoreKey aanwezig zijn.

assert stores_df["StoreKey"].notna().all(),  "StoreKey bevat één of meerdere ontbrekende waarden."
#Controleert of er rijen zijn waarbij de StoreKey ontbreekt.

assert stores_df.loc[stores_df["Status"] == "On","CloseReason"].isna().all(), "Eén of meerdere actieve winkels hebben een CloseReason."
#Controleert of er actieve winkels zijn met een CloseReason.

assert stores_df.loc[stores_df["Status"] == "Off","CloseReason"].notna().all(), "Eén of meerdere gesloten winkels hebben geen CloseReason."
#Controleert of er gesloten winkels zijn zonder CloseReason.

# 6 ***Analyse Promotions data***
In dit onderdeel wordt de promotions dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data.

### 6.1 Data importeren

In [109]:
#Inlezen Promotion dataset.
promotions_df = load_data("Contoso_Promotion.csv")

### 6.2 Data analyse

In [110]:
#Intiële analyse met behulp van init_analyse functie
init_analyse(promotions_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DiscountPercent  28 non-null     float64
 1   Year             28 non-null     int64  
 2   Quarter          28 non-null     str    
 3   Month            28 non-null     str    
 4   Day              28 non-null     int64  
 5   PromotionKey     28 non-null     int64  
 6   PromotionLabel   28 non-null     int64  
 7   PromotionName    28 non-null     str    
 8   Year.1           28 non-null     int64  
 9   Quarter.1        28 non-null     str    
 10  Month.1          28 non-null     str    
 11  Day.1            28 non-null     int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 4.1 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


DiscountPercent    0
Year               0
Quarter            0
Month              0
Day                0
PromotionKey       0
PromotionLabel     0
PromotionName      0
Year.1             0
Quarter.1          0
Month.1            0
Day.1              0
dtype: int64


Aantal unieke waarden:


DiscountPercent     6
Year                4
Quarter             4
Month               6
Day                 2
PromotionKey       28
PromotionLabel     28
PromotionName      10
Year.1              4
Quarter.1           4
Month.1             7
Day.1               1
dtype: int64


Eerste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
0,0.05,2011,Qtr 1,March,31,2,2,North America Spring Promotion,2011,Qtr 1,January,1
1,0.20,2011,Qtr 2,April,30,6,6,Asian Spring Promotion,2011,Qtr 1,February,1
2,0.07,2011,Qtr 2,April,30,8,8,European Spring Promotion,2011,Qtr 1,February,1
3,0.10,2011,Qtr 2,June,30,7,7,Asian Summer Promotion,2011,Qtr 2,May,1
4,0.10,2011,Qtr 3,September,30,3,3,North America Back-to-School Promotion,2011,Qtr 3,July,1
5,0.10,2011,Qtr 3,September,30,9,9,European Back-to-Scholl Promotion,2011,Qtr 3,August,1
6,0.20,2011,Qtr 4,December,31,4,4,North America Holiday Promotion,2011,Qtr 4,November,1
7,0.15,2012,Qtr 1,January,31,5,5,Asian Holiday Promotion,2011,Qtr 4,November,1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1



Laatste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1
10,0.20,2012,Qtr 2,April,30,15,15,Asian Spring Promotion,2012,Qtr 1,February,1
11,0.07,2012,Qtr 2,April,30,17,17,European Spring Promotion,2012,Qtr 1,February,1
12,0.10,2012,Qtr 2,June,30,16,16,Asian Summer Promotion,2012,Qtr 2,May,1
13,0.10,2012,Qtr 3,September,30,12,12,North America Back-to-School Promotion,2012,Qtr 3,July,1
14,0.10,2012,Qtr 3,September,30,18,18,European Back-to-Scholl Promotion,2012,Qtr 3,August,1
15,0.20,2012,Qtr 4,December,31,13,13,North America Holiday Promotion,2012,Qtr 4,November,1
16,0.15,2013,Qtr 1,January,31,14,14,Asian Holiday Promotion,2012,Qtr 4,November,1
17,0.20,2013,Qtr 1,January,31,19,19,European Holiday Promotion,2012,Qtr 4,October,1


DiscountPercent is in tegenstelling tot voorgaande datasets met een . voor decimalen aangegeven. 
Er zijn 2 datum waarden aanwezig in de set. Een eerste analyse lijkt er op te wijzen dat dit een begin en eind datum van de promotie is, waarbij de datums eindigend op .1 de startdatums zijn. 
Door deze om te zetten naar datums kunnen we dit later gebruiken om te analyseren hoeveel er gebruik gemaakt is van de promoties in de periode.
Eventueel vergelijkingen maken tussen periodes met en zonder promotie en de winstgevendheid hiervan.  
Het aantal promotionkeys komt overeen met het aantal in de sales set. Promotionlabel lijkt altijd gelijk te zijn aan de key, onduidelijk waarom.  
Er zitten ook waarden in de dataset met betrekking tot 2014, deze kunnen niet gerelateerd worden aan de salesdata gezien deze loopt tot 31 december 2023.
Promotion names lijken grotendeels per jaar overeen te komen. Effectiviteit over de afgelopen 3 jaar zou dus bekeken kunnen worden. 
Eventueel waarde "European Back-to-Scholl Promotion" opschonen naar School.

Key 1 lijkt een doorlopende promotie te zijn om verkopen met geen korting vast te leggen.
Is er correlatie tussen promotion en discountamount? 
De volgende stap is om de samenhang met de salesdata te analyseren. Er is hier een aansluiting op de SalesKey.



In [111]:
#Controleren welke promotionkeys in de salesset zitten en vice versa.
display(promotions_df["PromotionKey"].isin(datesales["PromotionKey"]).value_counts())
display(datesales["PromotionKey"].isin(promotions_df["PromotionKey"]).value_counts())

PromotionKey
True    28
Name: count, dtype: int64

PromotionKey
True    2282482
Name: count, dtype: int64

In [112]:
#Controleren of alle gegevens met promotionkey 1 geen discount hebben. 
datesales.loc[datesales["PromotionKey"] == 1, "DiscountAmount"].sum()

np.float64(0.0)

In [113]:
#Alle waarden voor promo 2 opzoeken en sorteren op index en in aparte dataframe zetten.
promo2 = datesales.loc[datesales["PromotionKey"]==2].sort_index()

In [114]:
#Controleren discountamount op basis van promotionkey 2.
#Verwachting DiscountPct is 5% van aanschafwaarde
promo2["SalesBeforeDiscount"] = promo2["UnitPrice"]*promo2["SalesQuantity"]
promo2["DiscountPct"] = promo2["DiscountAmount"]/promo2["SalesBeforeDiscount"]*100
#Geen directe aansluiting tussen percentage en discountamount.
#Vermoeden UnitPrice*DiscountQuantity* DiscountPercent = DiscountAmount
promo2["DiscountPercent"] = promo2["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Discountpercentage koppelen aan waarde in promo dataframe. Zoek promotion key in promo en match deze op promotion dataset, voegt verolgens de discountpercent toe als kolom.
promo2["DiscountControle"] = promo2["UnitPrice"]*promo2["DiscountQuantity"]*promo2["DiscountPercent"]
#Discount herrekenen ter controle
promo2a = promo2.loc[(promo2["DiscountControle"].round(2) != promo2["DiscountAmount"].round(2)),["DiscountAmount","DiscountControle"]]
promo2a["ControleDiscountAmount"] = promo2a["DiscountAmount"] - promo2a["DiscountControle"]
display(promo2a.head(50))
display(promo2a[(promo2a["ControleDiscountAmount"]<-0.001)|(promo2a["ControleDiscountAmount"]>0.001)])
#Controleren afwijkingen, lijkt alleen op afrondingen terecht te komen.

,DiscountAmount,DiscountControle,ControleDiscountAmount
DateKey,,,
2011-01-01,4.49,4.48,0.00
2011-01-01,34.48,34.49,-0.00
2011-01-01,8.74,8.75,-0.00
2011-01-01,4.49,4.48,0.00
2011-01-01,4.04,4.03,0.00
2011-01-01,4.49,4.48,0.00
2011-01-01,0.57,0.58,-0.00
2011-01-01,65.97,65.98,-0.00
2011-01-02,9.07,9.08,-0.00


,DiscountAmount,DiscountControle,ControleDiscountAmount
DateKey,,,


Op basis van PromotionKey 2 is het verband tussen de PromotionKey het discountpercentage in de promotions dataset en de DiscountAmount in de sales data set geanalyseerd.  
De UnitPrice * DiscountQuantity * DiscountPercent is gelijk aan DiscountAmount voor deze promocode.  
Op basis hiervan lijken er specifieke voorwaarden aan promotieacties te zitten over de hoeveelheden waarop korting wordt geleverd.  
De volgende actie is om deze controle voor alle promotioncodes uit te voeren in de salesdataset.

In [115]:
#Kopieeren van salesdataset
promocontrole = datesales.copy()
#Mappen van discountpercentage
promocontrole["DiscountPercent"] = promocontrole["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Berekenen discountamount
promocontrole["DiscountControle"] = promocontrole["UnitPrice"]*promocontrole["DiscountQuantity"]*promocontrole["DiscountPercent"]
#Controleren waarden niet gelijk aan 0
promocontrole["ControleDiscountAmount"] = promocontrole["DiscountAmount"]-promocontrole["DiscountControle"]
display(promocontrole[promocontrole["ControleDiscountAmount"] != 0])
#Controleren afrondingsverschillen.
display(promocontrole[(promocontrole["ControleDiscountAmount"]<-0.001)|(promocontrole["ControleDiscountAmount"]>0.001)])

,SalesKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-12-15,686,1,226,1260,10,18.86,36.99,4,0,0.00,1,7.40,75.44,140.56,0.20,7.40,-0.00
2011-10-11,3394,1,250,1139,10,150.84,328.00,4,0,0.00,1,65.60,603.36,"1,246.40",0.20,65.60,-0.00
2011-12-01,4301,1,263,70,10,22.05,47.95,4,0,0.00,1,9.59,88.20,182.21,0.20,9.59,-0.00
2011-11-06,4678,1,226,1272,10,3.54,6.95,4,0,0.00,1,1.39,14.16,26.41,0.20,1.39,-0.00
2011-10-17,8615,1,304,1423,10,86.91,189.00,4,0,0.00,1,37.80,347.64,718.20,0.20,37.80,-0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011-01-15,3177099,3,200,536,2,50.47,99.00,32,3,297.00,9,44.55,"1,615.04","3,123.45",0.05,44.55,-0.00
2011-11-25,3196138,2,199,128,4,73.11,143.40,18,3,430.20,3,86.04,"1,315.98","2,495.16",0.20,86.04,-0.00
2013-03-09,3279273,4,309,341,26,444.69,967.00,60,3,"2,901.00",14,947.66,"25,347.33","57,072.34",0.07,947.66,-0.00


,SalesKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,


Nu testen we dit voor de gehele dataset.

In [116]:
#Herhalen stappen voor controle
datesales["DiscountPercent"] = datesales["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
datesales["DiscountControle"] = datesales["UnitPrice"]*datesales["DiscountQuantity"]*datesales["DiscountPercent"]
datesales["ControleDiscountAmount"] = datesales["DiscountAmount"]-datesales["DiscountControle"]

#Dataframe weergeven waar controle waarde niet gelijk is aan 0.
display(datesales[datesales["ControleDiscountAmount"] != 0])

#Dataframe met waardes met een afwijking groter dan een duizendste achter de decimaal.
display(datesales[(datesales["ControleDiscountAmount"]<-0.001)|(datesales["ControleDiscountAmount"]>0.001)])

,SalesKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-12-15,686,1,226,1260,10,18.86,36.99,4,0,0.00,1,7.40,75.44,140.56,0.20,7.40,-0.00
2011-10-11,3394,1,250,1139,10,150.84,328.00,4,0,0.00,1,65.60,603.36,"1,246.40",0.20,65.60,-0.00
2011-12-01,4301,1,263,70,10,22.05,47.95,4,0,0.00,1,9.59,88.20,182.21,0.20,9.59,-0.00
2011-11-06,4678,1,226,1272,10,3.54,6.95,4,0,0.00,1,1.39,14.16,26.41,0.20,1.39,-0.00
2011-10-17,8615,1,304,1423,10,86.91,189.00,4,0,0.00,1,37.80,347.64,718.20,0.20,37.80,-0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011-01-15,3177099,3,200,536,2,50.47,99.00,32,3,297.00,9,44.55,"1,615.04","3,123.45",0.05,44.55,-0.00
2011-11-25,3196138,2,199,128,4,73.11,143.40,18,3,430.20,3,86.04,"1,315.98","2,495.16",0.20,86.04,-0.00
2013-03-09,3279273,4,309,341,26,444.69,967.00,60,3,"2,901.00",14,947.66,"25,347.33","57,072.34",0.07,947.66,-0.00


,SalesKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,


Bovenstaande controle bevestigt het vermoeden dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan DiscountAmount.
Er lijkt dus een verband te zijn tussen promotion en de hoeveelheid items. 
Promotions kunnen bijvoorbeeld zijn vanaf een bepaalde hoeveelheid of bij een bepaalde hoeveelheid een product gratis. Het is niet helemaal duidelijk voor iedere promotie wat deze verbanden zijn en of deze relevant zijn.

In [117]:
#Kijken naar eerste rijen met DiscountQuantity en PromotionKey 2.
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().head(20)
#Laate 50 waarden
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().tail(20)

,SalesKey,ChannelKey,StoreKey,ProductKey,PromotionKey,UnitCost,UnitPrice,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,DiscountPercent,DiscountControle,ControleDiscountAmount
DateKey,,,,,,,,,,,,,,,,,
2011-03-31,708402,3,200,1059,2,155.43,338.00,9,0,0.00,2,33.80,"1,398.87","3,008.20",0.05,33.80,-0.00
2011-03-31,805417,3,200,1525,2,133.19,402.00,9,0,0.00,2,40.20,"1,198.71","3,577.80",0.05,40.20,0.00
2011-03-31,916643,3,200,864,2,23.45,50.99,9,0,0.00,2,5.10,211.05,453.81,0.05,5.10,0.00
2011-03-31,794844,4,308,1628,2,6.39,13.89,9,0,0.00,3,2.08,57.51,122.93,0.05,2.08,-0.00
2011-03-31,2569110,1,69,1387,2,13.33,28.99,9,0,0.00,2,2.90,119.97,258.01,0.05,2.90,0.00
2011-03-31,346278,4,308,821,2,13.75,29.90,9,0,0.00,2,2.99,123.75,266.11,0.05,2.99,0.00
2011-03-31,115601,4,308,1586,2,5.82,12.66,9,0,0.00,2,1.27,52.38,112.67,0.05,1.27,0.00
2011-03-31,921326,4,308,256,2,208.52,409.00,9,0,0.00,4,81.80,"1,876.68","3,599.20",0.05,81.80,-0.00
2011-03-31,2705434,1,149,412,2,195.24,382.95,9,0,0.00,2,38.30,"1,757.16","3,408.26",0.05,38.30,0.00


In [118]:
#Kijken naar ratio's verkoop hoeveelheid afgezet tegen hoeveelheid discounts.
datesales.loc[datesales["PromotionKey"]==2].groupby("SalesQuantity")["DiscountQuantity"].describe()

,count,mean,std,min,25%,50%,75%,max
SalesQuantity,,,,,,,,
4,"40,938.00",2.00,1.41,0.00,1.00,2.00,3.00,4.00
6,101.00,2.19,1.44,0.00,1.00,2.00,3.00,4.00
8,"4,480.00",3.30,2.05,0.00,2.00,3.00,5.00,8.00
9,"68,654.00",2.00,1.41,0.00,1.00,2.00,3.00,4.00
12,"3,031.00",2.85,2.29,0.00,1.00,3.00,4.00,12.00
16,931.00,4.55,2.62,0.00,3.00,4.00,6.00,14.00
18,"7,286.00",3.31,2.06,0.00,2.00,3.00,5.00,8.00
20,29.00,10.00,2.96,3.00,8.00,10.00,12.00,15.00
24,563.00,4.81,2.69,0.00,3.00,5.00,7.00,12.00


Voor nu lijkt er verder geen logisch verband te zijn tussen promotion acties en wanneer deze wel of niet wordt gegeven en op welke hoeveelheid producten.

#### Voorlopige Conclusie
De promotieset bevat data over 28 promoties die open vanaf 1 januari 2011 tot en met 31 maart 2014.
De Promotion met Key 1 lijkt de standaardwaarde te zijn voor verkopen zonder enige korting.
Het verband met de salesset is het feit dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan de DiscountAmount.

Er is geen duidelijk verband te vinden tussen de verschillende promotieacties en hoeveelheden. De verwachting zou zijn dat kortingen gelden vanaf een aantal items of per aantal items worden toegepast. Dit verband is echter niet te herleiden.

**To-do**:
- De set heeft 2 datumwaarden. De eerste heeft betrekking op de einddatum de tweede op de begindatum van de promotion. Deze dienen nog duidelijk benoemd te worden. 
- Er zit een fout in de omschrijving van de Europese back-to-scholl actie. Deze nog aanpassen naar "European Back-to-School Promotion".

## 6.3 Data cleaning
Aan de hand van de to-do list wordt de data opgeschoond.

In [119]:
#Dataframe met waardes met een afwijking groter dan een duizendste achter de decimaal.
promotions_df["EndMonth"] = pd.to_datetime(promotions_df["Month"],format="%B").dt.month.astype("int64")
#Kolom verplaatsen naar begin met rest van data.
kolom = promotions_df.pop("EndMonth")
promotions_df.insert(3,"EndMonth", kolom)

promotions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DiscountPercent  28 non-null     float64
 1   Year             28 non-null     int64  
 2   Quarter          28 non-null     str    
 3   EndMonth         28 non-null     int64  
 4   Month            28 non-null     str    
 5   Day              28 non-null     int64  
 6   PromotionKey     28 non-null     int64  
 7   PromotionLabel   28 non-null     int64  
 8   PromotionName    28 non-null     str    
 9   Year.1           28 non-null     int64  
 10  Quarter.1        28 non-null     str    
 11  Month.1          28 non-null     str    
 12  Day.1            28 non-null     int64  
dtypes: float64(1), int64(7), str(5)
memory usage: 4.3 KB


In [120]:
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.
promotions_df["PromotionEndDate"] = pd.to_datetime({"year":promotions_df["Year"],"month":promotions_df["EndMonth"],"day":promotions_df["Day"]})


In [121]:
#Startmaand converteren naar integer
promotions_df["StartMonth"] = pd.to_datetime(promotions_df["Month.1"],format="%B").dt.month.astype("int64")
#Op basis van jaar, maand en, dag gegevens einddatum als datumwaarde aanmaken in de set.
promotions_df["PromotionStartDate"] = pd.to_datetime({"year":promotions_df["Year.1"],"month":promotions_df["StartMonth"],"day":promotions_df["Day.1"]})

In [122]:
#Kolomnamen hernoemen naar duidelijk benamingen die eind en begin aangeven.
promotions_df = promotions_df.rename(columns={"Year": "EndYear",
                              "Month": "EndMonthstr",
                              "Quarter": "EndQuarter",
                              "Day": "EndDay",
                              "Year.1":"StartYear",
                              "Quarter.1":"StartQuarter",
                              "Month.1":"StartMonthstr",
                              "Day.1":"StartDay"})

In [123]:
#Corrigeren foutieve schrijfwize European Back-to-School Promotion
promotions_df.loc[promotions_df["PromotionName"]== "European Back-to-Scholl Promotion","PromotionName"] = "European Back-to-School Promotion"

In [124]:
#Verplaatsen StartMonth
kolom = promotions_df.pop("StartMonth")
promotions_df.insert(11,"StartMonth", kolom)

De eind-, en startdatumwaarde zijn duidelijk benoemd binnen de dataset.
Deze zijn ook geconverteerd tot bruikbare datevalues. Deze maken ook duidelijk dat de laatste promotion met 0% op het moment tot en met  eind 2014 loopt. 
De datum set zal hier op aangepast worden. 
Verder is de foutieve schrijfwijze van de European Back-to-School promotion aangepast.

## 6.4 Validatie

In [125]:
assert promotions_df["PromotionKey"].is_unique, "PromotionKey bevat één of meerdere duplicaten."
# Controleert of alle PromotionKeys uniek zijn.

assert promotions_df["PromotionKey"].notna().all(), "PromotionKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere promotie een PromotionKey bevat.

assert promotions_df["DiscountPercent"].between(0, 1).all(), "DiscountPercent bevat één of meerdere waarden buiten het bereik van 0 tot 1."
# Controleert of alle kortingspercentages tussen 0% en 100% liggen.

assert pd.api.types.is_datetime64_any_dtype(promotions_df["PromotionStartDate"]), "PromotionStartDate is niet omgezet naar een geldig datetime datatype."
# Controleert of PromotionStartDate correct is omgezet naar een datumwaarde.

assert pd.api.types.is_datetime64_any_dtype(promotions_df["PromotionEndDate"]), "PromotionEndDate is niet omgezet naar een geldig datetime datatype."
# Controleert of PromotionEndDate correct is omgezet naar een datumwaarde.

assert (promotions_df["PromotionStartDate"]<= promotions_df["PromotionEndDate"]).all(), "Eén of meerdere promoties hebben een startdatum na de einddatum."
# Controleert of iedere promotie begint voor of op de bijbehorende einddatum.

# 7 ***Analyse Product (Sub)Category***

## 7.1 Data importeren

In [126]:
#Importeren product category
productcat_df = load_data("Contoso_Product_Category.csv")

## 7.2 Data analyse

In [127]:
#Initiële analyse met behulp van init_analyse functie
init_analyse(productcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ProductCategory     8 non-null      str  
 1   ProductCategoryKey  8 non-null      int64
dtypes: int64(1), str(1)
memory usage: 377.0 bytes

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategory       0
ProductCategoryKey    0
dtype: int64


Aantal unieke waarden:


ProductCategory       8
ProductCategoryKey    8
dtype: int64


Eerste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2



Laatste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Dataset lijkt een subcategory te zijn van een andere dataset. 
Vermoedelijk de Product Sub Category dataset.
Verbinding lijkt te zijn ProductKey -> ProductSubCategroy -> ProductCategory
Eerst product Sub Category bekijken.

In [128]:
#Importeren product subcategory
productsubcat_df = load_data("Contoso_ProductSubcategory.csv")

In [129]:
#Initiële analyse met behulp van init_analyse functie
init_analyse(productsubcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   ProductCategoryKey     44 non-null     int64
 1   ProductSubcategory     44 non-null     str  
 2   ProductSubcategoryKey  44 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.8 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategoryKey       0
ProductSubcategory       0
ProductSubcategoryKey    0
dtype: int64


Aantal unieke waarden:


ProductCategoryKey        8
ProductSubcategory       44
ProductSubcategoryKey    44
dtype: int64


Eerste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
1,1,Bluetooth Headphones,6
2,1,Headphones,5
3,1,MP4&MP3,1
4,1,Radio,3
5,1,Recorder,2
6,1,Recording Pen,4
7,1,Speakers,7
8,2,Car Video,13
9,2,Home Theater System,11



Laatste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
24,4,Digital SLR Cameras,24
25,4,Film Cameras,25
26,5,Cell phones Accessories,33
27,5,Home & Office Phones,29
28,5,Smart phones & PDAs,32
29,5,Touch Screen Phones,31
30,6,Audio Books,36
31,6,Movie DVD,35
32,6,Music CD,34
33,7,Boxed Games,38


Er lijken 44 unieke Subcategorieën te zijn, verbonden aan 1 van de 8 hoofdcategorieën.
Volgende stap is om te kijken hoeveel van de subcategorykeys zich in de productlijst bevinden.

De header ProductSubcategory is niet in consistente camelcase

In [130]:
#filter die controleert welke keys wel of niet in de product dataset zitten.
masker = productsubcat_df["ProductSubcategoryKey"].isin(product_df["ProductSubcategoryKey"])
#Geeft alle waarden weer die wel in de dataset zitten.
print(f"{productsubcat_df.loc[masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën in product dataframe")
display(productsubcat_df[masker])

print(f"{productsubcat_df.loc[~masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën niet in product dataframe")
display(productsubcat_df[~masker])

22 Productsubcategorieën in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
1,1,Bluetooth Headphones,6
3,1,MP4&MP3,1
6,1,Recording Pen,4
8,2,Car Video,13
9,2,Home Theater System,11
10,2,Televisions,9
12,2,VCD & DVD,10
14,3,Computers Accessories,22
15,3,Desktops,17
16,3,Laptops,15


22 Productsubcategorieën niet in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
2,1,Headphones,5
4,1,Radio,3
5,1,Recorder,2
7,1,Speakers,7
11,2,TV & Video Accessories,14
13,3,Computer Setup & Service,21
18,3,Netbooks,16
25,4,Film Cameras,25
30,6,Audio Books,36


Dataset lijkt met namen hardware en multimedia apparaten te betreffen.  De 22 waarden sluiten ook aan met de unique count van de productset.

De productgroep die niet in de productcategory voorkomt heeft met name berekking op witgoed, en media.

In [131]:
#Controleren of alle keys in de productcategoryKey zitten.
productsubcat_df["ProductCategoryKey"].isin(productcat_df["ProductCategoryKey"])
productcat_df["ProductCategoryKey"].isin(productsubcat_df["ProductCategoryKey"])

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
Name: ProductCategoryKey, dtype: bool

In [132]:
#Controleren welke productcategorieën voorkomen in products
print("Categorieën in product data set:")
display(productcat_df[(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])])

#Controleren welke productcategorieën niet voorkomen in products
print("Categorieën niet in product data set:")
productcat_df[~(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])]


Categorieën in product data set:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Categorieën niet in product data set:


,ProductCategory,ProductCategoryKey
4,Games and Toys,7
5,Home Appliances,8


Conform de hypothese zijn witgoed en games en speelgoed uitgesloten van de set.

#### Voorlopige conclusie

De dataset lijkt 22 van de 44 subcategorieën te gebruiken.
Daarnaast zijn de categoriën met ProductCategoryKey 7 en 8 niet van toepassing op de sales dataset.

*To-do*
- Productcategory en subcategory mergen voor snellere toegang.

## 7.3 Data cleaning

### Mergen productcategory en productsubcategory

In [133]:
#Mergen van dataframees met behulp van pandas merge functie.
productmerge_df = productsubcat_df.merge(productcat_df, on="ProductCategoryKey")

## 7.4 Validatie

In [134]:
assert productcat_df["ProductCategoryKey"].is_unique, "ProductCategoryKey bevat één of meerdere duplicaten."
# Controleert of alle ProductCategoryKeys uniek zijn.

assert productcat_df["ProductCategoryKey"].notna().all(), "ProductCategoryKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere productcategorie een ProductCategoryKey bevat.

assert productsubcat_df["ProductSubcategoryKey"].is_unique, "ProductSubcategoryKey bevat één of meerdere duplicaten."
# Controleert of alle ProductSubcategoryKeys uniek zijn.

assert productsubcat_df["ProductSubcategoryKey"].notna().all(),  "ProductSubcategoryKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere productsubcategorie een ProductSubcategoryKey bevat.

assert productsubcat_df["ProductCategoryKey"].notna().all(), "ProductCategoryKey bevat één of meerdere ontbrekende waarden in de subcategorie dataset."
# Controleert of iedere subcategorie aan een productcategorie gekoppeld is.

assert productsubcat_df["ProductCategoryKey"].isin( productcat_df["ProductCategoryKey"]).all(), "Eén of meerdere ProductCategoryKeys uit ProductSubCategory bestaan niet in ProductCategory."
# Controleert of iedere ProductCategoryKey uit de subcategorie dataset voorkomt in de ProductCategory dataset.

# 8 ***Analyse Channel data***

## 8.1 Data importeren

In [135]:
#Importeren channel dataset
channel_df = load_data("Contoso_Channel.csv")

## 8.2 Data analyse

In [136]:
#Initiële analyse met behulp van init_analyse functie
init_analyse(channel_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Channel      4 non-null      int64
 1   ChannelName  4 non-null      str  
dtypes: int64(1), str(1)
memory usage: 222.0 bytes

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


Channel        0
ChannelName    0
dtype: int64


Aantal unieke waarden:


Channel        4
ChannelName    4
dtype: int64


Eerste 20 rijen:


,Channel,ChannelName
0,1,Store
1,2,Online
2,3,Catalog
3,4,Reseller



Laatste 20 rijen:


,Channel,ChannelName
0,1,Store
1,2,Online
2,3,Catalog
3,4,Reseller


De Channel dataset lijkt weinig gegevens te bevatten, enkel over de aanwezige kanalen.
In alle andere datasets wordt de key specifiek benoemd, hier staat enkel "Channel".
Dit dient nog gewijzigd te worden om consistent te blijven met de andere datasets.

## 8.3 Data Cleaning

In [137]:
#Channel hernoemd naar ChannelKey.
channel_df = channel_df.rename(columns={"Channel":"ChannelKey"})

## 8.4 Validatie

In [138]:
assert channel_df["ChannelKey"].is_unique, "ChannelKey bevat één of meerdere duplicaten."
# Controleert of alle ChannelKeys uniek zijn.

assert channel_df["ChannelKey"].notna().all(),  "ChannelKey bevat één of meerdere ontbrekende waarden."
# Controleert of iedere rij een ChannelKey bevat.

Dit onderdeel is bedoeld om onderlinge regels tussen de verschillende datasets te evalueren, alvorens deze op te slaan als parquet-bestanden die in PowerBI geladen zullen worden.

# 9 ***GDP per capita/ BBP per hoofd per jaar toevoegen(Worldbank dataset)***

Van de worldbank is een dataset gedownload met betrekking tot gegevens van jaar tot jaar over GDP per capita, oftewel bruto binnenlands product per hoofd van de bevolking.  
Het meet indirect hoe welvarend een land is.
Met deze gegevens kan er vergeleken worden hoe sterk omzet afhankelijk is van de welvaart van een land. 


## 9.1 Importen van data

In [139]:
#Importeren GDP dataset verkregen via World Bank Group
GDP_df = load_data("Contoso_GDP.csv",
                     skiprows=4)
#Eerste 4 rijen betreffen geen relevante data

## 9.2 Initiële analyse

In [140]:
#Initiële analyse met behulp van init_analyse functie
init_analyse(GDP_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    265 non-null    str    
 1   Country Code    265 non-null    str    
 2   Indicator Name  265 non-null    str    
 3   Indicator Code  265 non-null    str    
 4   1960            151 non-null    float64
 5   1961            154 non-null    float64
 6   1962            156 non-null    float64
 7   1963            156 non-null    float64
 8   1964            156 non-null    float64
 9   1965            162 non-null    float64
 10  1966            163 non-null    float64
 11  1967            166 non-null    float64
 12  1968            167 non-null    float64
 13  1969            167 non-null    float64
 14  1970            189 non-null    float64
 15  1971            190 non-null    float64
 16  1972            190 non-null    float64
 17  1973            190

Country Name        0
Country Code        0
Indicator Name      0
Indicator Code      0
1960              114
                 ... 
2022                9
2023               14
2024               18
2025               32
Unnamed: 70       265
Length: 71, dtype: int64


Aantal unieke waarden:


Country Name      265
Country Code      265
Indicator Name      1
Indicator Code      1
1960              149
                 ... 
2022              255
2023              249
2024              245
2025              231
Unnamed: 70         0
Length: 71, dtype: int64


Eerste 20 rijen:


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"28,440.04","30,082.16","30,654.49","22,664.37","26,827.34","31,000.57","34,897.62","38,590.57",NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.09,186.91,197.37,225.40,208.96,226.84,...,"1,529.92","1,553.62","1,508.03","1,351.50","1,560.89","1,675.90","1,571.13","1,628.23","1,722.39",NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.47,491.34,496.60,510.79,356.50,357.26,413.76,416.87,NaN,NaN
3,Africa Western and Central,AFW,GDP per capita (current US$),NY.GDP.PCAP.CD,121.94,127.45,133.82,139.00,148.55,155.59,...,"1,577.20","1,723.11","2,219.41","2,034.44","2,116.94","2,143.07","1,846.25","1,416.23","1,600.06",NaN
4,Angola,AGO,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"2,832.15","2,891.83","2,507.87","1,749.18","2,266.97","3,598.54","2,885.51","2,720.82","3,129.48",NaN
5,Albania,ALB,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"5,006.36","5,897.65","6,069.44","6,027.91","7,242.46","7,756.96","9,740.70","11,374.01","12,998.15",NaN
6,Andorra,AND,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"40,672.99","42,819.77","41,257.80","37,361.09","42,425.70","42,414.06","46,812.45","49,357.44","54,291.50",NaN
7,Arab World,ARB,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,212.90,210.82,225.82,243.91,256.20,...,"6,105.92","6,708.53","6,683.67","5,749.75","6,708.21","7,960.81","7,504.36","7,602.41","7,544.41",NaN
8,United Arab Emirates,ARE,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"43,733.62","47,135.35","45,938.61","37,991.74","44,118.49","50,759.76","49,850.69","50,273.51",NaN,NaN
9,Argentina,ARG,GDP per capita (current US$),NY.GDP.PCAP.CD,778.25,971.34,870.22,852.97,"1,176.20","1,281.83",...,"14,532.50","11,752.80","9,955.97","8,535.60","10,738.02","13,962.19","14,261.85","13,969.78","14,898.09",NaN



Laatste 20 rijen:


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
245,Tanzania,TZA,GDP per capita (current US$),NY.GDP.PCAP.CD,275.30,285.16,304.00,329.01,346.30,342.08,...,986.67,"1,023.11","1,063.32","1,117.42","1,159.86","1,207.52","1,224.00","1,192.61","1,318.82",NaN
246,Uganda,UGA,GDP per capita (current US$),NY.GDP.PCAP.CD,55.38,56.18,55.50,61.96,68.63,99.95,...,765.22,792.17,822.17,845.77,882.79,963.07,"1,002.31","1,077.91","1,206.30",NaN
247,Ukraine,UKR,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"2,604.29","3,059.05","3,619.86","3,709.77","4,775.95","4,199.67","5,139.60","5,391.62","5,865.88",NaN
248,Upper middle income,UMC,GDP per capita (current US$),NY.GDP.PCAP.CD,147.18,144.91,140.74,150.56,167.97,183.44,...,"7,676.40","8,149.60","8,244.31","7,949.26","9,409.61","9,796.27","10,118.62","10,443.35","10,890.10",NaN
249,Uruguay,URY,GDP per capita (current US$),NY.GDP.PCAP.CD,491.03,603.97,659.39,586.82,744.65,705.20,...,"19,184.66","19,249.94","18,315.74","15,757.53","17,881.81","21,009.29","23,378.64","24,308.50","25,215.82",NaN
250,United States,USA,GDP per capita (current US$),NY.GDP.PCAP.CD,"2,999.86","3,059.16","3,236.01","3,366.37","3,565.31","3,818.29",...,"60,047.72","62,875.67","65,227.96","64,465.30","71,441.23","78,008.69","82,586.78","86,169.66","90,026.52",NaN
251,Uzbekistan,UZB,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,"2,271.47","1,894.09","2,128.66","2,087.96","2,370.36","2,698.60","3,015.98","3,337.45","3,968.27",NaN
252,St. Vincent and the Grenadines,VCT,GDP per capita (current US$),NY.GDP.PCAP.CD,154.84,163.04,166.50,154.86,164.44,166.16,...,"7,988.04","8,428.13","8,740.59","8,351.20","8,641.28","9,693.56","10,582.37","11,501.23","12,561.77",NaN
253,"Venezuela, RB",VEN,GDP per capita (current US$),NY.GDP.PCAP.CD,939.20,954.04,"1,006.56","1,060.09",873.60,869.04,...,"3,791.63","3,421.51","2,523.12","1,506.04","2,004.94","3,155.04","3,617.47","4,244.46","3,494.81",NaN
254,British Virgin Islands,VGB,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Er zitten een hoop kolommen in de set die we niet nodig hebben in verband met de historische data over voorgaande jaren.
Deze dienen we eerst te beperken voordat we verder gaan. 

In [141]:
#Dataframe creeëren met alleen relevante waarde voor de dataset.
GDP_df = GDP_df[["Country Name","Country Code","Indicator Name","Indicator Code","2011","2012","2013"]]

In [142]:
#Dataframe "smaller" maken door year en GDP waarde als kolom op te slaan.
GDP_df = GDP_df.melt(
   id_vars=["Country Name", "Country Code"],
   value_vars=["2011", "2012", "2013"],
   var_name="Year",
    value_name="GDPPerCapita"
)


In [143]:
#Jaar omzetten naar integer type.
GDP_df["Year"] = GDP_df["Year"].astype("int64")

In [144]:
#Controleren of waarden in Geography overeenkomen met waarden in GDP dataset.
geography_df2["RegionCountryName"].isin(GDP_df["Country Name"]).value_counts()

RegionCountryName
True     644
False     27
Name: count, dtype: int64

In [145]:
#Controleren welke unieke waarden in GDP dataset niet voorkomen in de geography dataset.
display(geography_df2[~geography_df2["RegionCountryName"].isin(GDP_df["Country Name"])]["RegionCountryName"].unique())

<ArrowStringArray>
[           'Iran',      'Kyrgyzstan',     'South Korea',           'Syria',
          'Taiwan', 'the Netherlands',          'Russia']
Length: 7, dtype: str

Lijst van landen die niet direct in de dataset gevonden worden.

In [146]:
#Lijst maken met ontbrekende landen
test = list(geography_df2[~(geography_df2["RegionCountryName"].isin(GDP_df["Country Name"]))]["RegionCountryName"].unique())

#Lijst joinen tot een string met een pipe tussen de woorden voor een or statement in str.contains.
test = "|".join(test)

#Kijken of er landen in de dataset zitten die delen van het woord bevatten.
GDP_df[GDP_df["Country Name"].str.contains(test)]["Country Name"]


111      Iran, Islamic Rep.
201      Russian Federation
226    Syrian Arab Republic
376      Iran, Islamic Rep.
466      Russian Federation
491    Syrian Arab Republic
641      Iran, Islamic Rep.
731      Russian Federation
756    Syrian Arab Republic
Name: Country Name, dtype: str

Drie landen gevonden die anders benoemd worden:
- Iran -> Iran, Islamic Rep.
- Russia -> Russian Federation
- Syria -> Syrian Arab Republic

Resteert 
- Kyrgyzstan
- South Korea
- Taiwan
- The Netherlands

In [147]:
#Controleren of reseterende landen op andere wijze zijn omschreven in de geography set.
restant="Kyrgy|Korea|Taiwan|Nether"
#Lijst met waarden die nog ontbreken.
display(GDP_df[GDP_df["Country Name"].str.contains(restant)]["Country Name"])
#Controleren op resterende waardes.

121              Kyrgyz Republic
125                  Korea, Rep.
175                  Netherlands
192    Korea, Dem. People's Rep.
386              Kyrgyz Republic
390                  Korea, Rep.
440                  Netherlands
457    Korea, Dem. People's Rep.
651              Kyrgyz Republic
655                  Korea, Rep.
705                  Netherlands
722    Korea, Dem. People's Rep.
Name: Country Name, dtype: str

2 waarden die Korea bevatten, Korea, Dem. People's Rep. betreft Noord Korea en bevat ook geen data.  
Er zijn waarden gevonden voor alles behalve Taiwan. Taiwan staat ook bekend als Republic of China.  
Hier nog een extra check op doen. 

In [148]:
#Kijken of republic of china wel aanwezig is.
display(GDP_df[GDP_df["Country Name"].str.contains("China")]["Country Name"])

40                    China
95     Hong Kong SAR, China
145        Macao SAR, China
305                   China
360    Hong Kong SAR, China
410        Macao SAR, China
570                   China
625    Hong Kong SAR, China
675        Macao SAR, China
Name: Country Name, dtype: str

Alleen voor Taiwan hebben we geen gegevens. De andere landen kunnen we koppelen, gezien de omvang van de dataset is dit acceptabel.


##  9.3 Data cleaning

### Key maken om GDP aan geography te koppelen.

In [149]:
#Aanmaken CountryKey als primairy key voor GDP dataset.
#Dictionary op basis waarvan de key wordt aangemaakt voor de landen die anders zijn omschreven in de GDP dataset.
mapping = {"Kyrgyz Republic":"Kyrgyzstan",
           "Iran, Islamic Rep.":"Iran",
           "Russian Federation":"Russia",
           "Syrian Arab Republic":"Syria",
           "Korea, Rep.":"South Korea",
           "Netherlands":"the Netherlands",
           "Taiwan":pd.NA
           }

#Vervang de namen in de huidige GDP Dataframe met de gekoppelde namen in de dictionary.
GDP_df["CountryKey"] = GDP_df["Country Name"].replace(mapping)

#Controleren welke waarden niet voorkomen in GDP. Enkel nog Taiwan zoals verwacht
display(geography_df2[~geography_df2["RegionCountryName"].isin(GDP_df["CountryKey"])])

,ContinentName,GeographyKey,GeographyType,RegionCountryName
13,Asia,281,Country/Region,Taiwan
36,Asia,323,State/Province,Taiwan
107,Asia,729,City,Taiwan


De originele namen zijn gemapt aan de waarden uit de geography dataset om deze als key te gebruiken. 
Vervolgens is er een nieuwe CountryKey kolom gemaakt om te linken op RegionCountryName.

## 9.4 Validatie

In [150]:
#Validates

assert GDP_df["CountryKey"].notna().all(), "CountryKey bevat ontbrekende waarden."
#Controleren of alle waarden van CountryKey gevuld zijn

assert not GDP_df.duplicated(["CountryKey", "Year"]).any(), "GDP bevat dubbele combinaties van land en jaar."
#Controleren of er dubbele combinaties van land en jaar in de dataset aanwezig zijn.

# 10 ***Integrale validatie***

In [151]:
#Integrale validaties
assert sales_df2["ProductKey"].isin(product_df["ProductKey"]).all(), "Niet alle ProductKeys uit Sales bestaan in Product."
# Controleert de relatie Sales -> Product.

assert sales_df2["DateKey"].isin(calendar_df2["DateKey"]).all(), "Niet alle DateKeys uit Sales bestaan in Calendar."
# Controleert de relatie Sales -> Calendar.

assert sales_df2["StoreKey"].isin(stores_df["StoreKey"]).all(), "Niet alle StoreKeys uit Sales bestaan in Stores."
# Controleert de relatie Sales -> Stores.

assert sales_df2["PromotionKey"].isin(promotions_df["PromotionKey"]).all(), "Niet alle PromotionKeys uit Sales bestaan in Promotions."
# Controleert de relatie Sales -> Promotions.

assert sales_df2["ChannelKey"].isin(channel_df["ChannelKey"]).all(), "Niet alle ChannelKeys uit Sales bestaan in Channel."
# Controleert de relatie Sales -> Channel.

assert stores_df["GeographyKey"].isin(geography_df2["GeographyKey"]).all(), "Niet alle GeographyKeys uit Stores bestaan in Geography."
# Controleert de relatie Stores -> Geography.

assert product_df["ProductSubcategoryKey"].isin(productmerge_df["ProductSubcategoryKey"]).all(), "Niet alle ProductSubcategoryKeys uit Product bestaan in ProductSubCategory."
# Controleert de relatie Product -> ProductSubCategory.


# 11 ***Opslaan***

In [ ]:
#Lokale opslage methode.
#Opslaan als parquet-bestanden.
#sales_df2.to_parquet(r"PQ/Contoso_sales.parquet")
#product_df.to_parquet(r"PQ/Contoso_product.parquet")
#calendar_df2.to_parquet(r"PQ/Contoso_calendar.parquet")
#geography_df2.to_parquet(r"PQ/Contoso_geography.parquet")
#stores_df.to_parquet(r"PQ/Contoso_stores.parquet")
#promotions_df.to_parquet(r"PQ/Contoso_promotions.parquet")
#productmerge_df.to_parquet(r"PQ/Contoso_productcat.parquet")
#channel_df.to_parquet(r"PQ/Contoso_channel.parquet")
#GDP_df.to_parquet(r"PQ/Contose_GDP.parquet")

In [ ]:
#Parquet bestanden uploaden na opschonen en valideren.
# SSH-verbinding voorbereiden
ssh = paramiko.SSHClient()

ssh.load_host_keys(
    str(Path.home() / ".ssh" / "known_hosts")
)

ssh.set_missing_host_key_policy(
    paramiko.RejectPolicy()
)

# Inloggen met beperkte uploadaccount
ssh.connect(
    hostname=HOST,
    username=USER,
    password=getpass("Voer het SFTP-wachtwoord in: "),
    look_for_keys=False,
    allow_agent=False
)

try:
    with ssh.open_sftp() as sftp:

        upload_parquet(
            sales_df2,
            "Contoso_sales.parquet",
            sftp
        )
        upload_parquet(product_df,
                       "Contoso_product.parquet",
                       sftp)
        upload_parquet(calendar_df2,
                       "Contoso_calendar.parquet",
                       sftp)
        upload_parquet(geography_df2,
                        "Contoso_geography.parquet",
                        sftp)
        upload_parquet(stores_df,
                       "Contoso_stores.parquet",
                       sftp)
        upload_parquet(promotions_df,
                       "Contoso_promotions.parquet",
                       sftp)
        upload_parquet(productmerge_df,
                       "Contoso_productcat.parquet",
                       sftp)
        upload_parquet(channel_df,
                       "Contoso_channel.parquet",
                       sftp)
        upload_parquet(GDP_df,
                       "Contoso_GDP.parquet",
                       sftp)

finally:
    ssh.close()

Succesvol geüpload: Contoso_sales.parquet
Rijen: 2,282,482
Bestandsgrootte: 46.16 MB
Succesvol geüpload: Contoso_product.parquet
Rijen: 1,690
Bestandsgrootte: 0.07 MB
Succesvol geüpload: Contoso_calendar.parquet
Rijen: 1,096
Bestandsgrootte: 0.02 MB
Succesvol geüpload: Contoso_geography.parquet
Rijen: 671
Bestandsgrootte: 0.01 MB
Succesvol geüpload: Contoso_stores.parquet
Rijen: 306
Bestandsgrootte: 0.01 MB
Succesvol geüpload: Contoso_promotions.parquet
Rijen: 28
Bestandsgrootte: 0.01 MB
Succesvol geüpload: Contoso_productcat.parquet
Rijen: 44
Bestandsgrootte: 0.00 MB
Succesvol geüpload: Contoso_channel.parquet
Rijen: 4
Bestandsgrootte: 0.00 MB
Succesvol geüpload: Contoso_GDP.parquet
Rijen: 795
Bestandsgrootte: 0.02 MB
